# Chapter 03. 텍스트를 숫자로 바꾸는 CountVectorizer와 TF-IDF

Chapter 02에서는 상품명을 단어 단위로 보고 빈도를 계산했다.
이번 Chapter에서는 도서 제목을 **머신러닝이 계산할 수 있는 숫자 형태**로 바꾼다.

사람은 다음 문장의 의미를 어느 정도 이해할 수 있다.

```
데이터 분석을 위한 파이썬
AI 시대의 머신러닝 입문
```

하지만 머신러닝 모델은 문자열 자체를 계산하지 못한다. 따라서 다음 변환이 필요하다.

> 텍스트 -> 단어 확인 -> 단어별 숫자 부여 -> 각 문서를 숫자 벡터로 표현

먼저 **Bag of Words**의 원리를 이해하고 `CountVectorizer`로 단어 등장 횟수를 숫자로 표현한다.
그 다음 모든 단어를 똑같이 중요하게 보는 방식의 한계를 확인하고,
`TfidfVectorizer`로 **문서 안에서는 자주 나오지만 전체 문서에서는 드문 단어**에 더 높은 중요도를 부여한다.

> 이번 Chapter에서 가장 중요한 것은 코드를 외우는 것이 아니다.
> **문장 하나가 어떻게 숫자 벡터 한 줄로 바뀌는지** 직접 확인하는 것이 핵심이다.

## 최종 결과물

```
chapter03.ipynb
chapter03_count_top_terms.csv
chapter03_tfidf_top_terms.csv
chapter03_count_matrix.npz   (선택)
chapter03_tfidf_matrix.npz   (선택)
```

---
# 2. 이번 Chapter의 핵심 질문

실행한 뒤 아래 질문에 **본인의 말로** 답할 수 있어야 한다.

1. 왜 텍스트를 숫자로 바꿔야 하는가?
2. Bag of Words는 무엇인가?
3. CountVectorizer가 만드는 숫자는 무엇을 의미하는가?
4. 행은 무엇이고 열은 무엇인가?
5. 같은 단어가 2번 나오면 숫자는 어떻게 되는가?
6. 모든 문서에 자주 등장하는 단어는 정말 중요한가?
7. TF는 무엇인가?
8. IDF는 무엇인가?
9. TF-IDF 값이 크다는 것은 무엇을 의미하는가?
10. CountVectorizer와 TfidfVectorizer는 어떤 차이가 있는가?

---
# 실습 1. Chapter 01 전처리 데이터 불러오기

## 해야 할 일

Chapter 01에서 만든 `book_bestseller_clean.csv`를 불러온다.

## 프롬프트 예시

```
Python과 pandas를 처음 배우고 있습니다.
현재 Notebook과 같은 폴더에
book_bestseller_clean.csv 파일이 있습니다.
pandas로 이 파일을 df_books라는 DataFrame으로 불러오고,
다음 내용을 확인하는 간단한 코드를 작성해 주세요.
1. 전체 행과 컬럼 개수
2. 컬럼 이름
3. 상품명 앞의 10개
4. 상품명 결측치 개수
CSV는 utf-8-sig 인코딩으로 저장했습니다.
초보자가 이해하기 쉽도록 작성해 주세요.
```

### 먼저 작업 폴더를 맞춘다

이 저장소의 VS Code 설정(`.vscode/settings.json`)에
`"jupyter.notebookFileRoot": "${workspaceFolder}"`가 들어 있다.
이 설정 때문에 Notebook을 실행하면 작업 폴더가 노트북이 있는 곳이 아니라
**저장소 루트**(`llm-data-analysis-course`)가 된다.

그래서 파일 이름만 적으면 `FileNotFoundError`가 난다.
아래 셀에서 작업 폴더를 `book-text-ml`로 맞춘 뒤 진행한다.

In [1]:
import os
from pathlib import Path

# 노트북이 있는 book-text-ml 폴더를 찾아 그쪽으로 이동한다.
# 폴더가 저장소 루트에 있든 notebooks 안에 있든 모두 찾는다.
if Path.cwd().name != "book-text-ml":
    for candidate in [Path("book-text-ml"), *Path(".").glob("*/book-text-ml")]:
        if candidate.is_dir():
            os.chdir(candidate)
            break

print("작업 폴더:", Path.cwd())
print("데이터 파일 있음:", Path("book_bestseller_clean.csv").exists())

작업 폴더: C:\dev\llm-data-analysis-course\notebooks\book-text-ml
데이터 파일 있음: True


In [2]:
import pandas as pd

DATA_PATH = "book_bestseller_clean.csv"
df_books = pd.read_csv(DATA_PATH, encoding="utf-8-sig")

print("데이터 크기:", df_books.shape)
print("컬럼:", df_books.columns.tolist())
print("상품명 결측치:", df_books["상품명"].isna().sum())
df_books[["상품명"]].head(10)

데이터 크기: (986, 8)
컬럼: ['순위', '판매상품ID', '상품명', '판매가', '저자', '출판사', '발행일', '분야']
상품명 결측치: 0


,상품명
0,"세네카, 오늘을 빼앗기고 있는 당신에게"
1,흔한남매 23
2,머니 트렌드 2027
3,싯다르타
4,한국사 이상현상 연구원(일반판)
5,테오
6,쇼펜하우어 인생수업(30만 부 기념 개정증보판)
7,니체의 초월자
8,판매의 법칙
9,마음의 어휘력


### 직접 확인할 것

- 파일이 정상적으로 열렸는가
- 상품명 컬럼이 존재하는가
- 한글이 정상적으로 보이는가
- Chapter 01 결과와 행 개수가 크게 달라지지 않았는가

---
# 실습 2. 분석할 제목 문자열 준비하기

## 해야 할 일

Vectorizer에 전달할 제목을 **문자열 형태**로 준비한다.

이번 Chapter에서는 우선 **원래 도서 제목 문자열을 그대로** 사용해 Vectorizer의 기본 원리를 이해한다.
Chapter 02의 형태소 분석 결과를 Vectorizer와 연결하는 방법은 실습 40에서 따로 본다.

## 프롬프트 예시

```
pandas DataFrame df_books에 '상품명' 컬럼이 있습니다.
이 컬럼을 Vectorizer에 넣을 수 있도록 문자열로 정리하고 싶습니다.
1. 결측치는 빈 문자열로 바꾸고
2. 모두 문자열로 통일하고
3. 앞뒤 공백을 제거하고
4. 내용이 없는 제목은 제외하고
5. 인덱스를 0부터 다시 매겨서 titles라는 Series로 만들어 주세요.
마지막에 제목 개수와 앞의 10개를 확인하는 코드도 넣어 주세요.
함수로 만들지 말고 바로 실행되는 코드로 작성해 주세요.
try/except 예외 처리는 넣지 마세요.
5줄 이내로 짧게 작성해 주세요.
설명은 코드 옆 짧은 주석으로만 달아 주세요.
```

In [3]:
titles = (
    df_books["상품명"]      # 도서 제목 컬럼
    .fillna("")             # 빈 값은 빈 문자열로
    .astype(str)            # 모두 문자열로 통일
    .str.strip()            # 앞뒤 공백 제거
)

# 내용이 없는 제목은 제외하고, 번호를 0부터 다시 매긴다
titles = titles[titles != ""].reset_index(drop=True)

print("사용할 제목 수:", len(titles))
titles.head(10)

사용할 제목 수: 986


0         세네카, 오늘을 빼앗기고 있는 당신에게
1                       흔한남매 23
2                   머니 트렌드 2027
3                          싯다르타
4             한국사 이상현상 연구원(일반판)
5                            테오
6    쇼펜하우어 인생수업(30만 부 기념 개정증보판)
7                       니체의 초월자
8                        판매의 법칙
9                       마음의 어휘력
Name: 상품명, dtype: str

---
# 실습 3. 머신러닝은 왜 텍스트를 숫자로 바꿀까?

머신러닝 알고리즘은 일반적으로 **숫자를 입력받아 계산**한다.

다음 같은 숫자는 계산할 수 있다.

```
[1, 0, 2, 0]
[0, 1, 0, 3]
```

하지만 다음 문자열을 그대로 더하거나 곱해서 학습할 수는 없다.

```
"데이터 분석을 위한 파이썬"
```

그래서 텍스트를 이렇게 바꾼다.

```
도서 제목
   ↓
사용된 단어 확인
   ↓
단어마다 열(column) 생성
   ↓
등장 여부 또는 등장 횟수를 숫자로 기록
   ↓
숫자 벡터
```

이 과정을 **텍스트 벡터화(Vectorization)** 라고 한다.

---
# 실습 4. 아주 작은 예제로 먼저 이해하기

## 해야 할 일

986개 제목을 바로 쓰면 행과 열이 많아 원리를 보기 어렵다.
먼저 **세 문장만** 사용한다.

## 프롬프트 예시

```
CountVectorizer를 처음 배우고 있습니다.
원리를 이해하기 위해 아주 짧은 예제 문장 3개를 리스트로 만들려고 합니다.
문장은 '파이썬 데이터 분석', '파이썬 머신러닝', '데이터 분석 입문'입니다.
sample_docs라는 변수에 담고 하나씩 출력하는 코드를 작성해 주세요.
함수로 만들지 말고 바로 실행되는 코드로 작성해 주세요.
try/except 예외 처리는 넣지 마세요.
5줄 이내로 짧게 작성해 주세요.
설명은 코드 옆 짧은 주석으로만 달아 주세요.
```

In [4]:
sample_docs = [
    "파이썬 데이터 분석",
    "파이썬 머신러닝",
    "데이터 분석 입문",
]

for doc in sample_docs:
    print(doc)

파이썬 데이터 분석
파이썬 머신러닝
데이터 분석 입문


사람이 직접 단어를 정리하면 다음과 같다.

```
파이썬  데이터  분석  머신러닝  입문
```

단어 순서를 `[데이터, 머신러닝, 분석, 입문, 파이썬]`으로 정했다고 하면,
첫 번째 문장 `파이썬 데이터 분석`은 다음처럼 표현할 수 있다.

```
[1, 0, 1, 0, 1]
```

이 숫자는 **의미 점수가 아니라 각 단어가 몇 번 등장했는지**를 나타낸다.

---
# 실습 5. Bag of Words 이해하기

앞의 방식은 **단어를 하나의 주머니에 넣고 몇 번 등장했는지 세는 것**과 비슷하다.
이 개념을 **Bag of Words(BoW)** 라고 부른다.

> 문장 -> 단어 순서보다 **어떤 단어가 몇 번 등장했는지**에 초점

예를 들어 다음 두 문장을 보자.

```
파이썬 데이터 분석
데이터 파이썬 분석
```

단어 순서는 다르지만 같은 단어가 같은 횟수로 등장하므로
Bag of Words 방식에서는 **동일한 Count 벡터**가 된다.

### 기억할 점

Bag of Words는 간단하고 유용하지만
**단어 순서와 문맥 정보를 충분히 표현하지 못한다.**
이번 과정에서는 먼저 이 단순한 표현을 이해한 뒤 분류와 추천에 활용한다.

---
# 실습 6. CountVectorizer 설치 확인하기

`CountVectorizer`는 scikit-learn에 포함되어 있다.
설치가 필요하면 다음을 실행한다.

```
python -m pip install scikit-learn
```

아래 셀이 오류 없이 실행되면 사용할 준비가 된 것이다.

## 프롬프트 예시

```
scikit-learn의 CountVectorizer를 사용하려고 합니다.
현재 환경에 설치되어 있는지 Notebook에서 확인하고 싶습니다.
import가 되는지와 scikit-learn 버전을 출력하는 코드를 작성해 주세요.
함수로 만들지 말고 바로 실행되는 코드로 작성해 주세요.
try/except 예외 처리는 넣지 마세요.
5줄 이내로 짧게 작성해 주세요.
설명은 코드 옆 짧은 주석으로만 달아 주세요.
```

In [5]:
from sklearn.feature_extraction.text import CountVectorizer

import sklearn
print("scikit-learn 버전:", sklearn.__version__)
print("CountVectorizer 사용 가능")

scikit-learn 버전: 1.9.0
CountVectorizer 사용 가능


---
# 실습 7. 작은 예제에 CountVectorizer 적용하기

## 해야 할 일

`fit_transform()` 한 번에 두 가지 동작이 수행된다.

| 동작 | 의미 |
|---|---|
| `fit` | 문서에서 **어떤 단어를 열로 사용할지** 학습 |
| `transform` | 각 문서를 **숫자 벡터로 변환** |

## 프롬프트 예시

```
문장 3개가 든 리스트 sample_docs가 있습니다.
CountVectorizer를 만들고 fit_transform()으로 숫자 행렬을 만들고 싶습니다.
만들어진 행렬의 크기(문서 수, 단어 수)도 함께 출력해 주세요.
함수로 만들지 말고 바로 실행되는 코드로 작성해 주세요.
try/except 예외 처리는 넣지 마세요.
5줄 이내로 짧게 작성해 주세요.
설명은 코드 옆 짧은 주석으로만 달아 주세요.
```

In [6]:
count_vectorizer = CountVectorizer()
X_count_sample = count_vectorizer.fit_transform(sample_docs)

print("행렬 크기:", X_count_sample.shape)   # (문서 수, 단어 수)

행렬 크기: (3, 5)


---
# 실습 8. 생성된 단어 사전 확인하기

## 해야 할 일

Vectorizer가 **어떤 단어를 열로 만들었는지** 확인한다.
숫자 벡터를 해석할 때는 반드시 **feature 이름의 순서와 함께** 봐야 한다.

## 프롬프트 예시

```
CountVectorizer로 fit_transform()을 이미 실행했습니다.
Vectorizer가 어떤 단어를 열로 만들었는지 확인하고 싶습니다.
get_feature_names_out()을 사용해 단어 목록을 출력하는 코드를 작성해 주세요.
함수로 만들지 말고 바로 실행되는 코드로 작성해 주세요.
try/except 예외 처리는 넣지 마세요.
5줄 이내로 짧게 작성해 주세요.
설명은 코드 옆 짧은 주석으로만 달아 주세요.
```

In [7]:
feature_names = count_vectorizer.get_feature_names_out()
print(feature_names)

['데이터' '머신러닝' '분석' '입문' '파이썬']


---
# 실습 9. 단어-문서 행렬 확인하기

## 해야 할 일

작은 예제이므로 행렬 전체를 배열로 바꿔 본다.

| 위치 | 의미 |
|---|---|
| 행(row) | 문서 또는 도서 제목 |
| 열(column) | 단어 |
| 값(value) | 해당 문서에서 단어가 등장한 횟수 |

In [8]:
X_count_sample.toarray()

array([[1, 0, 1, 0, 1],
       [0, 1, 0, 0, 1],
       [1, 0, 1, 1, 0]])

## 프롬프트 예시

```
Python과 pandas를 처음 배우고 있습니다.
CountVectorizer로 만든 X_count_sample과 count_vectorizer 객체가 이미 있습니다.
sample_docs는 문장 3개가 든 리스트입니다.
단어-문서 행렬을 pandas DataFrame으로 만들어
행 이름은 원래 문장, 열 이름은 단어가 되게 하고 싶습니다.
함수로 만들지 말고 바로 실행되는 코드로 작성해 주세요.
try/except 예외 처리는 넣지 마세요.
5줄 이내로 짧게 작성해 주세요.
설명은 코드 옆 짧은 주석으로만 달아 주세요.
```

In [9]:
sample_count_df = pd.DataFrame(
    X_count_sample.toarray(),
    columns=feature_names,
    index=sample_docs,
)
sample_count_df

,데이터,머신러닝,분석,입문,파이썬
파이썬 데이터 분석,1,0,1,0,1
파이썬 머신러닝,0,1,0,0,1
데이터 분석 입문,1,0,1,1,0


### 반드시 한 행을 직접 읽어보기

`파이썬 데이터 분석` 행에서 다음을 직접 확인한다.

```
파이썬   -> 1
데이터   -> 1
분석     -> 1
머신러닝 -> 0
입문     -> 0
```

이 해석이 되면 CountVectorizer의 핵심을 이해한 것이다.

---
# 실습 10. 같은 단어가 여러 번 나오면 어떻게 될까?

## 해야 할 일

같은 단어가 두 번 나오는 문장을 넣어 값이 어떻게 되는지 확인한다.

## 프롬프트 예시

```
CountVectorizer가 같은 단어가 여러 번 나올 때 값을 어떻게 기록하는지 확인하고 싶습니다.
'파이썬 파이썬 데이터'와 '데이터 분석' 두 문장으로 테스트하려고 합니다.
CountVectorizer를 적용하고 결과를 DataFrame 표로 보여 주세요.
행 이름은 원래 문장, 열 이름은 단어로 해주세요.
함수로 만들지 말고 바로 실행되는 코드로 작성해 주세요.
try/except 예외 처리는 넣지 마세요.
5줄 이내로 짧게 작성해 주세요.
설명은 코드 옆 짧은 주석으로만 달아 주세요.
```

In [10]:
repeat_docs = [
    "파이썬 파이썬 데이터",
    "데이터 분석",
]

repeat_vectorizer = CountVectorizer()
X_repeat = repeat_vectorizer.fit_transform(repeat_docs)

repeat_df = pd.DataFrame(
    X_repeat.toarray(),
    columns=repeat_vectorizer.get_feature_names_out(),
    index=repeat_docs,
)
repeat_df

,데이터,분석,파이썬
파이썬 파이썬 데이터,1,0,2
데이터 분석,1,1,0


첫 번째 문장에서 `파이썬`이 두 번 등장했으므로 값이 **2**가 된다.

즉 CountVectorizer는 단순한 존재 여부가 아니라 **기본적으로 등장 횟수를 기록한다.**

---
# 실습 11. CountVectorizer의 기본 토큰 기준 확인하기

## 해야 할 일

`CountVectorizer()`는 기본 설정에서 문자열을 내부적으로 나누어 토큰을 만든다.
기본 토큰 패턴은 일반적으로 **두 글자 이상**의 단어 문자를 대상으로 한다.
따라서 한 글자 토큰은 기본 설정에서 **제외될 수 있다.**

## 프롬프트 예시

```
CountVectorizer의 기본 토큰 기준을 확인하고 싶습니다.
'AI 데이터 분석 R 파이썬'이라는 문장 하나로 테스트해서
만들어진 단어 목록을 출력해 주세요.
한 글자인 'R'이 결과에서 빠지는지도 확인하는 코드를 넣어 주세요.
함수로 만들지 말고 바로 실행되는 코드로 작성해 주세요.
try/except 예외 처리는 넣지 마세요.
5줄 이내로 짧게 작성해 주세요.
설명은 코드 옆 짧은 주석으로만 달아 주세요.
```

In [11]:
test_docs = [
    "AI 데이터 분석 R 파이썬",
]

test_vectorizer = CountVectorizer()
X_test = test_vectorizer.fit_transform(test_docs)

print(test_vectorizer.get_feature_names_out())
print()
print("'R'이 빠졌는지 확인:", "r" not in test_vectorizer.get_feature_names_out())

['ai' '데이터' '분석' '파이썬']

'R'이 빠졌는지 확인: True


### 왜 중요한가?

Chapter 02에서 최소 글자수 조건을 정했던 것처럼,
Vectorizer 역시 **어떤 문자열을 단어로 볼지에 따라 결과가 달라진다.**

따라서 기본 설정을 무조건 정답으로 생각하지 않는다.

---
# 실습 12. 실제 도서 제목에 CountVectorizer 적용하기

## 해야 할 일

이제 실제 상품명 986개에 적용하고 행렬 크기를 확인한다.

| 값 | 의미 |
|---|---|
| `X_count.shape[0]` | 사용한 도서 제목 수 |
| `X_count.shape[1]` | CountVectorizer가 만든 단어 수 |

## 프롬프트 예시

```
도서 제목이 든 pandas Series titles가 있습니다.
CountVectorizer를 적용해 X_count라는 행렬을 만들고,
단어 목록은 count_terms에 담고 싶습니다.
문서 수, 단어 수, 행렬 크기를 출력하는 코드도 함께 작성해 주세요.
함수로 만들지 말고 바로 실행되는 코드로 작성해 주세요.
try/except 예외 처리는 넣지 마세요.
5줄 이내로 짧게 작성해 주세요.
설명은 코드 옆 짧은 주석으로만 달아 주세요.
```

In [12]:
count_vectorizer = CountVectorizer()
X_count = count_vectorizer.fit_transform(titles)
count_terms = count_vectorizer.get_feature_names_out()

print("문서 수:", X_count.shape[0])
print("단어 수:", X_count.shape[1])
print("행렬 크기:", X_count.shape)

문서 수: 986
단어 수: 2179
행렬 크기: (986, 2179)


---
# 실습 13. vocabulary_ 확인하기

## 해야 할 일

Vectorizer 내부에는 **단어와 열 번호의 대응 정보**가 들어 있다.

> **주의:** 이 숫자는 빈도가 아니라 **행렬에서 그 단어가 위치한 열 번호**다.

| 대상 | 숫자의 의미 |
|---|---|
| `vocabulary_`의 값 | 열 위치 |
| 행렬의 셀 값 | 단어 등장 횟수 |

## 프롬프트 예시

```
CountVectorizer의 vocabulary_ 속성을 확인하고 싶습니다.
앞의 20개 항목만 출력하는 코드를 작성해 주세요.
함수로 만들지 말고 바로 실행되는 코드로 작성해 주세요.
try/except 예외 처리는 넣지 마세요.
5줄 이내로 짧게 작성해 주세요.
설명은 코드 옆 짧은 주석으로만 달아 주세요.
```

In [13]:
list(count_vectorizer.vocabulary_.items())[:20]

[('세네카', 1174),
 ('오늘을', 1473),
 ('빼앗기고', 1055),
 ('있는', 1671),
 ('당신에게', 633),
 ('흔한남매', 2170),
 ('23', 74),
 ('머니', 822),
 ('트렌드', 1969),
 ('2027', 63),
 ('싯다르타', 1313),
 ('한국사', 2061),
 ('이상현상', 1613),
 ('연구원', 1446),
 ('일반판', 1650),
 ('테오', 1946),
 ('쇼펜하우어', 1212),
 ('인생수업', 1638),
 ('30만', 96),
 ('기념', 452)]

## 프롬프트 예시

```
CountVectorizer의 vocabulary_ 값이 '빈도'가 아니라 '열 번호'라는 것을
직접 비교해서 확인하고 싶습니다.
단어 하나를 골라 그 단어의 vocabulary_ 값과
행렬에서 그 열의 합계(전체 등장 횟수)를 나란히 출력해 주세요.
함수로 만들지 말고 바로 실행되는 코드로 작성해 주세요.
try/except 예외 처리는 넣지 마세요.
5줄 이내로 짧게 작성해 주세요.
설명은 코드 옆 짧은 주석으로만 달아 주세요.
```

In [14]:
# 혼동하지 않도록 직접 비교해 본다
sample_term = count_terms[100]
col = count_vectorizer.vocabulary_[sample_term]

print("단어:", sample_term)
print("vocabulary_ 값(열 번호):", col)
print("이 열의 전체 등장 횟수 합계:", int(X_count[:, col].sum()))

단어: 365
vocabulary_ 값(열 번호): 100
이 열의 전체 등장 횟수 합계: 2


---
# 실습 14. 희소 행렬(Sparse Matrix) 이해하기

## 해야 할 일

전체 제목을 벡터화하면 **대부분의 값이 0**이 된다.
어떤 제목에 `파이썬`이 없다면 그 열의 값은 0이기 때문이다.

```
[0, 0, 1, 0, 0, 0, 0, 1, 0, ...]
```

0이 매우 많은 데이터를 일반 배열로 모두 저장하면 메모리를 낭비한다.
그래서 scikit-learn의 Vectorizer는 기본적으로 **희소 행렬**을 반환한다.

## 프롬프트 예시

```
CountVectorizer가 반환한 X_count가 희소 행렬인지 확인하고 싶습니다.
행렬의 타입과 0이 아닌 값의 개수를 출력하는 코드를 작성해 주세요.
함수로 만들지 말고 바로 실행되는 코드로 작성해 주세요.
try/except 예외 처리는 넣지 마세요.
5줄 이내로 짧게 작성해 주세요.
설명은 코드 옆 짧은 주석으로만 달아 주세요.
```

In [15]:
print("행렬 타입:", type(X_count))
print("저장된 0이 아닌 값의 개수:", X_count.nnz)

행렬 타입: <class 'scipy.sparse._csr.csr_matrix'>
저장된 0이 아닌 값의 개수: 3744


### 중요한 주의

작은 예제에서는 다음 코드가 괜찮다.

```python
X_count_sample.toarray()
```

하지만 **전체 행렬은 무조건 이렇게 바꾸지 않는다.**

```python
# 전체 데이터에서는 주의
X_count.toarray()
```

데이터가 커지면 많은 메모리를 사용할 수 있다.
전체를 Dense 배열로 바꾸기보다 **필요한 일부 행이나 집계값만** 확인하는 습관을 들인다.

---
# 실습 15. 실제 데이터의 첫 번째 제목 벡터 확인하기

## 해야 할 일

첫 번째 도서 제목에서 **0보다 큰 단어만** 찾아 원래 제목과 비교한다.

In [16]:
print(titles.iloc[0])

세네카, 오늘을 빼앗기고 있는 당신에게


## 프롬프트 예시

```
CountVectorizer로 만든 X_count와 단어 목록 count_terms가 있습니다.
첫 번째 문서에서 0보다 큰 단어만 찾아
(단어, 등장횟수) 쌍의 리스트로 보고 싶습니다.
전체를 toarray()로 바꾸지 말고 희소 행렬의 indices와 data를 사용해 주세요.
함수로 만들지 말고 바로 실행되는 코드로 작성해 주세요.
try/except 예외 처리는 넣지 마세요.
5줄 이내로 짧게 작성해 주세요.
설명은 코드 옆 짧은 주석으로만 달아 주세요.
```

In [17]:
first_row = X_count.getrow(0)
indices = first_row.indices
values = first_row.data

first_title_terms = [
    (count_terms[index], value)
    for index, value in zip(indices, values)
]
first_title_terms

[('세네카', np.int64(1)),
 ('오늘을', np.int64(1)),
 ('빼앗기고', np.int64(1)),
 ('있는', np.int64(1)),
 ('당신에게', np.int64(1))]

### 직접 검증

- 원래 제목에 실제로 존재하는 단어인가?
- 등장 횟수가 맞는가?
- 예상하지 못한 토큰이 만들어졌는가?
- 한 글자 단어가 빠졌는가?
- 숫자나 영문은 어떻게 처리되었는가?

**Vectorizer의 결과를 직접 읽어보는 것이 중요하다.**

---
# 실습 16. 전체 데이터에서 많이 등장한 단어 확인하기

## 해야 할 일

Count 행렬의 **각 열을 합하면** 전체 문서에서 각 단어가 등장한 총 횟수가 된다.

## 프롬프트 예시

```
CountVectorizer로 만든 X_count와 단어 목록 count_terms가 있습니다.
각 열을 합해서 전체 문서에서 단어가 등장한 총 횟수를 구하고,
'단어'와 '전체등장횟수' 두 컬럼을 가진 DataFrame으로 만들어
많이 등장한 순서로 정렬한 뒤 상위 30개를 보고 싶습니다.
함수로 만들지 말고 바로 실행되는 코드로 작성해 주세요.
try/except 예외 처리는 넣지 마세요.
5줄 이내로 짧게 작성해 주세요.
설명은 코드 옆 짧은 주석으로만 달아 주세요.
```

In [18]:
import numpy as np

count_sums = np.asarray(X_count.sum(axis=0)).ravel()

count_summary = pd.DataFrame({
    "단어": count_terms,
    "전체등장횟수": count_sums,
})
count_summary = count_summary.sort_values(
    "전체등장횟수",
    ascending=False,
).reset_index(drop=True)

count_summary.head(30)

,단어,전체등장횟수
0,2026,60
1,2027,56
2,에디션,28
3,해커스,25
4,기념,19
5,기본서,18
6,세트,18
7,2026년,17
8,토익,17
9,the,15


### Chapter 02 결과와 다를 수 있다

두 Chapter에서 **토큰화 조건이 다르기 때문**이다.

| Chapter 02 | 현재 기본 CountVectorizer |
|---|---|
| Kiwi 형태소 분석 | 자체 기본 토큰 기준 |
| 품사 필터링(NNG/NNP/SL) | 없음 |
| 최소 글자수 2 | 기본 패턴(두 글자 이상) |
| 불용어 제거 | 없음 |

따라서 결과 차이가 생기면 **어느 쪽이 틀렸다고 바로 판단하지 말고 전처리 조건을 비교한다.**

---
# 실습 17. Count 상위 단어 저장하기

## 프롬프트 예시

```
단어 빈도가 정리된 DataFrame count_summary가 있습니다.
상위 30개만 골라 chapter03_count_top_terms.csv로 저장하고 싶습니다.
Windows Excel에서 한글이 깨지지 않게 utf-8-sig 인코딩을 쓰고,
인덱스는 저장하지 말아 주세요.
저장이 되었는지 확인하는 코드도 넣어 주세요.
함수로 만들지 말고 바로 실행되는 코드로 작성해 주세요.
try/except 예외 처리는 넣지 마세요.
5줄 이내로 짧게 작성해 주세요.
설명은 코드 옆 짧은 주석으로만 달아 주세요.
```

In [19]:
count_top30 = count_summary.head(30)
count_top30.to_csv(
    "chapter03_count_top_terms.csv",
    index=False,
    encoding="utf-8-sig",
)

print("저장 완료:", Path("chapter03_count_top_terms.csv").exists())

저장 완료: True


In [20]:
pd.read_csv(
    "chapter03_count_top_terms.csv",
    encoding="utf-8-sig",
).head()

,단어,전체등장횟수
0,2026,60
1,2027,56
2,에디션,28
3,해커스,25
4,기념,19


---
# 실습 18. Count 방식의 한계 생각해 보기

CountVectorizer는 이해하기 쉽지만 한 가지 중요한 문제가 있다.

어떤 단어가 **거의 모든 문서에서 반복해서** 등장한다고 하자.

```
책   도서   이야기   세상
```

이런 단어는 전체 등장 횟수가 높을 수 있다.
하지만 **특정 문서를 다른 문서와 구분하는 데 반드시 중요한 단어라고 볼 수는 없다.**

반대로 어떤 단어가 특정 문서에서만 두드러지게 나온다면
그 문서의 특징을 더 잘 설명할 수 있다.

**이 생각에서 TF-IDF가 출발한다.**

---
# 실습 19. TF 이해하기

TF는 **Term Frequency**의 약자다. 초보자 단계에서는 이렇게 이해하면 충분하다.

> **한 문서 안에서 특정 단어가 얼마나 나타나는가?**

예를 들어 한 문서가 다음과 같다고 하자.

```
파이썬 파이썬 데이터 분석
```

`파이썬`은 두 번, `데이터`와 `분석`은 한 번 등장한다.
즉 같은 문서 안에서 `파이썬`의 빈도가 더 높다.

> 다만 scikit-learn의 TF-IDF 계산에는 **정규화와 IDF가 함께 적용**되므로
> 최종 TF-IDF 값을 단순한 등장 횟수 자체로 해석하면 안 된다.

---
# 실습 20. DF 이해하기

DF는 **Document Frequency**다.

> **특정 단어가 몇 개 문서에 등장하는가?**

문서가 100개 있고 `데이터`가 80개 문서에 나온다면 DF가 높다.
반대로 `약동학`이 2개 문서에만 나온다면 DF가 낮다.

DF는 단어가 **전체 문서에 얼마나 널리 퍼져 있는지** 보여준다.

---
# 실습 21. IDF 이해하기

IDF는 **Inverse Document Frequency**다. 핵심 생각은 다음과 같다.

| 단어 | 문서를 구분하는 힘 | IDF |
|---|---|---|
| 거의 모든 문서에 등장 | 상대적으로 작을 수 있음 | 작아짐 |
| 적은 수의 문서에 등장 | 구분에 도움이 될 수 있음 | 커질 수 있음 |

scikit-learn은 기본적으로 0으로 나누는 문제 등을 줄이기 위해
**smoothing이 포함된 IDF 공식**을 사용한다.

개념 학습 단계에서는 공식을 외우기보다,
**전체 문서에서 흔한 단어의 가중치를 낮추고 드문 단어의 가중치를 높이는 역할**로 이해한다.

---
# 실습 22. TF-IDF를 한 문장으로 정리하기

TF-IDF는 다음 두 관점을 결합한다.

| | 질문 |
|---|---|
| TF | 이 문서에서 얼마나 나타나는가? |
| IDF | 전체 문서에서는 얼마나 흔하거나 드문가? |

따라서 TF-IDF 값은 이렇게 생각할 수 있다.

> **이 문서에서는 눈에 띄지만 전체 문서에서는 너무 흔하지 않은 단어**
> -> 상대적으로 높은 TF-IDF를 가질 가능성

TF-IDF가 높다고 해서 **현실 세계에서 절대적으로 중요한 단어라는 뜻은 아니다.**
현재 문서 집합과 현재 Vectorizer 설정 안에서 **상대적으로 중요한 특징**이라는 뜻이다.

---
# 실습 23. 작은 예제로 TfidfVectorizer 적용하기

## 해야 할 일

같은 세 문장에 TF-IDF를 적용하고 Count 결과와 비교한다.

## 프롬프트 예시

```
문장 3개가 든 sample_docs가 있습니다.
TfidfVectorizer를 적용하고 결과를 DataFrame 표로 보고 싶습니다.
행 이름은 원래 문장, 열 이름은 단어로 하고,
값은 소수점 3자리까지만 보여 주세요.
함수로 만들지 말고 바로 실행되는 코드로 작성해 주세요.
try/except 예외 처리는 넣지 마세요.
5줄 이내로 짧게 작성해 주세요.
설명은 코드 옆 짧은 주석으로만 달아 주세요.
```

In [21]:
from sklearn.feature_extraction.text import TfidfVectorizer

tfidf_sample_vectorizer = TfidfVectorizer()
X_tfidf_sample = tfidf_sample_vectorizer.fit_transform(sample_docs)
tfidf_sample_terms = tfidf_sample_vectorizer.get_feature_names_out()

tfidf_sample_df = pd.DataFrame(
    X_tfidf_sample.toarray(),
    columns=tfidf_sample_terms,
    index=sample_docs,
)
tfidf_sample_df.round(3)

,데이터,머신러닝,분석,입문,파이썬
파이썬 데이터 분석,0.577,0.000,0.577,0.000,0.577
파이썬 머신러닝,0.000,0.796,0.000,0.000,0.605
데이터 분석 입문,0.518,0.000,0.518,0.681,0.000


## 프롬프트 예시

```
같은 문장 3개에 대한 Count 결과 DataFrame(sample_count_df)과
TF-IDF 결과 DataFrame(tfidf_sample_df)이 있습니다.
두 표를 나란히 출력해서 비교하고 싶습니다.
Jupyter에서 표 형태가 유지되도록 display()를 사용해 주세요.
함수로 만들지 말고 바로 실행되는 코드로 작성해 주세요.
try/except 예외 처리는 넣지 마세요.
5줄 이내로 짧게 작성해 주세요.
설명은 코드 옆 짧은 주석으로만 달아 주세요.
```

In [22]:
print("=== Count ===")
display(sample_count_df)

print("=== TF-IDF ===")
display(tfidf_sample_df.round(3))

=== Count ===


,데이터,머신러닝,분석,입문,파이썬
파이썬 데이터 분석,1,0,1,0,1
파이썬 머신러닝,0,1,0,0,1
데이터 분석 입문,1,0,1,1,0


=== TF-IDF ===


,데이터,머신러닝,분석,입문,파이썬
파이썬 데이터 분석,0.577,0.000,0.577,0.000,0.577
파이썬 머신러닝,0.000,0.796,0.000,0.000,0.605
데이터 분석 입문,0.518,0.000,0.518,0.681,0.000


### 비교할 것

- Count는 정수인가?
- TF-IDF는 소수인가?
- 모든 단어 값이 동일한가?
- 여러 문서에 반복되는 단어와 한 문서에만 나오는 단어의 값이 다른가?

---
# 실습 24. TfidfVectorizer가 만든 단어 확인하기

CountVectorizer와 마찬가지로 **열에는 단어가 배치된다.**
즉 TF-IDF도 최종적으로 같은 구조를 만든다.

```
행 = 문서
열 = 단어
값 = 해당 문서에서 그 단어의 TF-IDF 가중치
```

## 프롬프트 예시

```
TfidfVectorizer도 CountVectorizer와 같은 방식으로 열에 단어를 배치하는지
확인하고 싶습니다.
TfidfVectorizer가 만든 단어 목록을 출력해 주세요.
함수로 만들지 말고 바로 실행되는 코드로 작성해 주세요.
try/except 예외 처리는 넣지 마세요.
5줄 이내로 짧게 작성해 주세요.
설명은 코드 옆 짧은 주석으로만 달아 주세요.
```

In [23]:
print(tfidf_sample_vectorizer.get_feature_names_out())

['데이터' '머신러닝' '분석' '입문' '파이썬']


---
# 실습 25. 단어별 IDF 값 확인하기

## 해야 할 일

`TfidfVectorizer`가 학습한 IDF 값을 직접 확인한다.

### 해석할 때 주의

IDF 숫자의 **절대값 자체를 외울 필요는 없다.**
어떤 단어가 더 여러 문서에 퍼져 있는지,
그 단어의 IDF가 상대적으로 어떻게 달라지는지를 확인하는 것이 핵심이다.

## 프롬프트 예시

```
TfidfVectorizer를 fit한 상태입니다.
학습된 IDF 값을 단어와 함께 보고 싶습니다.
'단어'와 'IDF' 두 컬럼을 가진 DataFrame으로 만들고
IDF가 큰 순서로 정렬해 주세요.
함수로 만들지 말고 바로 실행되는 코드로 작성해 주세요.
try/except 예외 처리는 넣지 마세요.
5줄 이내로 짧게 작성해 주세요.
설명은 코드 옆 짧은 주석으로만 달아 주세요.
```

In [24]:
idf_df = pd.DataFrame({
    "단어": tfidf_sample_vectorizer.get_feature_names_out(),
    "IDF": tfidf_sample_vectorizer.idf_,
})
idf_df.sort_values("IDF", ascending=False)

,단어,IDF
1,머신러닝,1.693147
3,입문,1.693147
0,데이터,1.287682
2,분석,1.287682
4,파이썬,1.287682


`데이터`, `분석`, `파이썬`은 3개 문서 중 2개에 등장하고,
`머신러닝`, `입문`은 1개 문서에만 등장한다.
**적은 문서에만 나온 단어의 IDF가 더 크다.**

---
# 실습 26. 실제 도서 제목을 TF-IDF로 변환하기

## 프롬프트 예시

```
도서 제목이 든 titles에 TfidfVectorizer를 적용하고 싶습니다.
결과 행렬은 X_tfidf, 단어 목록은 tfidf_terms에 담아 주세요.
행렬 크기, 문서 수, 단어 수를 출력하는 코드도 함께 작성해 주세요.
함수로 만들지 말고 바로 실행되는 코드로 작성해 주세요.
try/except 예외 처리는 넣지 마세요.
5줄 이내로 짧게 작성해 주세요.
설명은 코드 옆 짧은 주석으로만 달아 주세요.
```

In [25]:
tfidf_vectorizer = TfidfVectorizer()
X_tfidf = tfidf_vectorizer.fit_transform(titles)
tfidf_terms = tfidf_vectorizer.get_feature_names_out()

print("TF-IDF 행렬 크기:", X_tfidf.shape)
print("문서 수:", X_tfidf.shape[0])
print("단어 수:", X_tfidf.shape[1])

TF-IDF 행렬 크기: (986, 2179)
문서 수: 986
단어 수: 2179


## 프롬프트 예시

```
CountVectorizer 결과 X_count와 TfidfVectorizer 결과 X_tfidf가 있습니다.
두 행렬의 크기가 같은지 확인하는 코드를 작성해 주세요.
함수로 만들지 말고 바로 실행되는 코드로 작성해 주세요.
try/except 예외 처리는 넣지 마세요.
5줄 이내로 짧게 작성해 주세요.
설명은 코드 옆 짧은 주석으로만 달아 주세요.
```

In [26]:
print("Count shape :", X_count.shape)
print("TF-IDF shape:", X_tfidf.shape)
print("같은 크기인가?:", X_count.shape == X_tfidf.shape)

Count shape : (986, 2179)
TF-IDF shape: (986, 2179)
같은 크기인가?: True


---
# 실습 27. 첫 번째 도서의 TF-IDF 주요 단어 확인하기

In [27]:
print("도서 제목:", titles.iloc[0])

도서 제목: 세네카, 오늘을 빼앗기고 있는 당신에게


## 프롬프트 예시

```
TfidfVectorizer를 fit한 상태입니다.
학습된 IDF 값을 단어와 함께 보고 싶습니다.
'단어'와 'IDF' 두 컬럼을 가진 DataFrame으로 만들고
IDF가 큰 순서로 정렬해 주세요.
함수로 만들지 말고 바로 실행되는 코드로 작성해 주세요.
try/except 예외 처리는 넣지 마세요.
5줄 이내로 짧게 작성해 주세요.
설명은 코드 옆 짧은 주석으로만 달아 주세요.
```

In [28]:
first_tfidf_row = X_tfidf.getrow(0)

first_tfidf_df = pd.DataFrame({
    "단어": tfidf_terms[first_tfidf_row.indices],
    "TF-IDF": first_tfidf_row.data,
})
first_tfidf_df = first_tfidf_df.sort_values(
    "TF-IDF",
    ascending=False,
).reset_index(drop=True)

first_tfidf_df

,단어,TF-IDF
0,빼앗기고,0.479241
1,세네카,0.452258
2,오늘을,0.452258
3,당신에게,0.452258
4,있는,0.395873


### 직접 확인할 것

- 원래 제목에 있는 단어만 나오는가?
- 가장 높은 단어는 무엇인가?
- 왜 그 단어의 값이 상대적으로 높을 수 있는가?
- 전체 문서에서 매우 흔한 단어는 값이 상대적으로 낮은가?

---
# 실습 28. 여러 도서의 주요 TF-IDF 단어 확인 함수 만들기

## 해야 할 일

반복해서 확인하기 위해 작은 함수를 만든다.
**데이터 행 수보다 큰 번호를 넣지 않도록 주의한다.**

## 프롬프트 예시

```
TF-IDF 행렬 X_tfidf, 단어 목록 tfidf_terms, 제목 Series titles가 있습니다.
도서 번호를 넣으면 그 도서의 TF-IDF 상위 단어를 보여주는
show_top_tfidf_terms(doc_index, top_n=5) 함수를 만들고 싶습니다.
도서 제목을 먼저 출력하고, 상위 단어 DataFrame을 반환하게 해주세요.
try/except 예외 처리는 넣지 마세요.
초보자가 읽기 쉽게 10줄 이내로 짧게 작성해 주세요.
설명은 코드 옆 짧은 주석으로만 달아 주세요.
```

In [29]:
def show_top_tfidf_terms(doc_index, top_n=5):
    row = X_tfidf.getrow(doc_index)
    result = pd.DataFrame({
        "단어": tfidf_terms[row.indices],
        "TF-IDF": row.data,
    })
    result = result.sort_values(
        "TF-IDF",
        ascending=False,
    ).head(top_n)
    print("도서 제목:", titles.iloc[doc_index])
    return result.reset_index(drop=True)

show_top_tfidf_terms(0, top_n=5)

도서 제목: 세네카, 오늘을 빼앗기고 있는 당신에게


,단어,TF-IDF
0,빼앗기고,0.479241
1,세네카,0.452258
2,오늘을,0.452258
3,당신에게,0.452258
4,있는,0.395873


In [30]:
show_top_tfidf_terms(10, top_n=5)

도서 제목: 수족관


,단어,TF-IDF
0,수족관,1.0


In [31]:
show_top_tfidf_terms(100, top_n=5)

도서 제목: 별이 빛나는 고양이 마을 5: 별빛 가득 달콤한 시간


,단어,TF-IDF
0,가득,0.374878
1,달콤한,0.374878
2,별빛,0.374878
3,별이,0.353771
4,빛나는,0.353771


---
# 실습 29. 전체 데이터에서 평균 TF-IDF가 높은 단어 확인하기

## 해야 할 일

각 단어의 TF-IDF 값을 전체 문서에서 평균내어 탐색한다.

### 주의

평균 TF-IDF가 높다는 사실만으로 그 단어가 **베스트셀러의 원인이라고 해석하면 안 된다.**
이 값은 현재 문서 집합에서 Vectorizer가 계산한 **텍스트 특징의 상대적 크기**다.

## 프롬프트 예시

```
TF-IDF 행렬 X_tfidf와 단어 목록 tfidf_terms가 있습니다.
각 단어의 TF-IDF 값을 전체 문서에서 평균내어
'단어'와 '평균_TFIDF' 컬럼을 가진 DataFrame으로 만들고
평균이 큰 순서로 정렬한 뒤 상위 30개를 보고 싶습니다.
함수로 만들지 말고 바로 실행되는 코드로 작성해 주세요.
try/except 예외 처리는 넣지 마세요.
5줄 이내로 짧게 작성해 주세요.
설명은 코드 옆 짧은 주석으로만 달아 주세요.
```

In [32]:
mean_tfidf = np.asarray(X_tfidf.mean(axis=0)).ravel()

tfidf_summary = pd.DataFrame({
    "단어": tfidf_terms,
    "평균_TFIDF": mean_tfidf,
})
tfidf_summary = tfidf_summary.sort_values(
    "평균_TFIDF",
    ascending=False,
).reset_index(drop=True)

tfidf_summary.head(30)

,단어,평균_TFIDF
0,2027,0.015500
1,2026,0.015419
2,에디션,0.009248
3,해커스,0.007563
4,2026년,0.006554
5,세트,0.006523
6,토익,0.006126
7,기념,0.005936
8,기본서,0.005839
9,the,0.005759


---
# 실습 30. TF-IDF 상위 단어 저장하기

## 프롬프트 예시

```
평균 TF-IDF가 정리된 DataFrame tfidf_summary가 있습니다.
상위 30개를 chapter03_tfidf_top_terms.csv로 저장하고 싶습니다.
utf-8-sig 인코딩을 쓰고 인덱스는 저장하지 말아 주세요.
저장 확인 코드도 넣어 주세요.
함수로 만들지 말고 바로 실행되는 코드로 작성해 주세요.
try/except 예외 처리는 넣지 마세요.
5줄 이내로 짧게 작성해 주세요.
설명은 코드 옆 짧은 주석으로만 달아 주세요.
```

In [33]:
tfidf_top30 = tfidf_summary.head(30)
tfidf_top30.to_csv(
    "chapter03_tfidf_top_terms.csv",
    index=False,
    encoding="utf-8-sig",
)

print("저장 완료:", Path("chapter03_tfidf_top_terms.csv").exists())

저장 완료: True


In [34]:
pd.read_csv(
    "chapter03_tfidf_top_terms.csv",
    encoding="utf-8-sig",
).head()

,단어,평균_TFIDF
0,2027,0.015500
1,2026,0.015419
2,에디션,0.009248
3,해커스,0.007563
4,2026년,0.006554


---
# 실습 31. Count 상위 단어와 TF-IDF 상위 단어 비교하기

## 프롬프트 예시

```
Count 상위 30개(count_top30)와 TF-IDF 상위 30개(tfidf_top30) DataFrame이 있습니다.
두 결과를 한 표에 나란히 놓고 비교하고 싶습니다.
컬럼은 Count 단어, Count 전체등장횟수, TFIDF 단어, 평균 TFIDF 네 개로 하고
상위 20행만 보여 주세요.
함수로 만들지 말고 바로 실행되는 코드로 작성해 주세요.
try/except 예외 처리는 넣지 마세요.
5줄 이내로 짧게 작성해 주세요.
설명은 코드 옆 짧은 주석으로만 달아 주세요.
```

In [35]:
comparison = pd.DataFrame({
    "Count_상위단어": count_top30["단어"].reset_index(drop=True),
    "Count_전체등장횟수": count_top30["전체등장횟수"].reset_index(drop=True),
    "TFIDF_상위단어": tfidf_top30["단어"].reset_index(drop=True),
    "평균_TFIDF": tfidf_top30["평균_TFIDF"].reset_index(drop=True),
})
comparison.head(20)

,Count_상위단어,Count_전체등장횟수,TFIDF_상위단어,평균_TFIDF
0,2026,60,2027,0.015500
1,2027,56,2026,0.015419
2,에디션,28,에디션,0.009248
3,해커스,25,해커스,0.007563
4,기념,19,2026년,0.006554
5,기본서,18,세트,0.006523
6,세트,18,토익,0.006126
7,2026년,17,기념,0.005936
8,토익,17,기본서,0.005839
9,the,15,the,0.005759


### 다음 질문에 답해 본다

- Count에서 높지만 TF-IDF에서는 상대적으로 낮아진 단어가 있는가?
- 두 목록 모두 높은 단어가 있는가?
- TF-IDF에서 새롭게 눈에 띄는 단어가 있는가?
- 그 차이를 **문서 빈도 관점**에서 설명할 수 있는가?

---
# 실습 32. 특정 단어가 몇 개 문서에 등장하는지 확인하기

## 해야 할 일

TF-IDF 차이를 이해하려면 **특정 단어의 문서 빈도(DF)** 를 직접 확인해 보는 것이 좋다.

| 결과 | 의미 |
|---|---|
| DF가 크다 | 많은 제목에서 등장 |
| DF가 작다 | 적은 수의 제목에서 등장 |

## 프롬프트 예시

```
CountVectorizer 객체 count_vectorizer와 행렬 X_count가 있습니다.
단어 하나를 넣으면 그 단어가 몇 개 문서에 등장하는지(DF) 돌려주는
document_frequency(term) 함수를 만들고 싶습니다.
단어 사전에 없는 단어는 0을 돌려주게 해주세요.
함수로 만들지 말고 바로 실행되는 코드로 작성해 주세요.
try/except 예외 처리는 넣지 마세요.
5줄 이내로 짧게 작성해 주세요.
설명은 코드 옆 짧은 주석으로만 달아 주세요.
```

In [36]:
def document_frequency(term):
    if term not in count_vectorizer.vocabulary_:
        return 0
    column_index = count_vectorizer.vocabulary_[term]
    column = X_count[:, column_index]
    return int((column > 0).sum())

print("데이터 DF:", document_frequency("데이터"))
print("파이썬 DF:", document_frequency("파이썬"))

데이터 DF: 0
파이썬 DF: 0


In [37]:
# Count 상위 단어들의 DF를 함께 확인한다
for term in count_top30["단어"].head(10):
    total = int(count_summary.loc[count_summary["단어"] == term, "전체등장횟수"].iloc[0])
    print(f"{term:>8} : 전체등장 {total:>4}회 / {document_frequency(term):>4}개 문서에 등장")

    2026 : 전체등장   60회 /   60개 문서에 등장
    2027 : 전체등장   56회 /   56개 문서에 등장
     에디션 : 전체등장   28회 /   28개 문서에 등장
     해커스 : 전체등장   25회 /   25개 문서에 등장
      기념 : 전체등장   19회 /   19개 문서에 등장
     기본서 : 전체등장   18회 /   18개 문서에 등장
      세트 : 전체등장   18회 /   18개 문서에 등장
   2026년 : 전체등장   17회 /   17개 문서에 등장
      토익 : 전체등장   17회 /   17개 문서에 등장
     the : 전체등장   15회 /   15개 문서에 등장


---
# 보충. Count와 TF-IDF의 차이를 숫자로 파고들기

## 해야 할 일

실습 31에서 두 상위 목록을 나란히 봤지만, **왜 순위가 달라지는지**는 아직 설명하지 못했다.
단어마다 등장 횟수, 문서 빈도(DF), IDF, 평균 TF-IDF를 한 표에 모아
순위가 크게 움직인 단어를 직접 찾아본다.

## 프롬프트 예시

```
Count 행렬 X_count, TF-IDF 행렬, 단어 목록 count_terms,
그리고 tfidf_vectorizer가 있습니다.
단어마다 등장횟수, 문서 빈도(DF), IDF, 평균 TF-IDF를 한 표에 모으고 싶습니다.
Count 기준 순위와 TF-IDF 기준 순위를 각각 매기고
두 순위의 차이를 '순위변동' 컬럼으로 만들어 주세요.
Count 순위가 높은 10개를 보여 주세요.
함수로 만들지 말고 바로 실행되는 코드로 작성해 주세요.
try/except 예외 처리는 넣지 마세요.
5줄 이내로 짧게 작성해 주세요.
설명은 코드 옆 짧은 주석으로만 달아 주세요.
```

In [38]:
# 1) 단어별 지표를 한 표에 모으기
doc_freq = np.asarray((X_count > 0).sum(axis=0)).ravel()   # 각 단어의 DF

term_table = pd.DataFrame({
    "단어": count_terms,
    "등장횟수": count_sums,
    "DF": doc_freq,
    "IDF": tfidf_vectorizer.idf_,
    "평균TFIDF": mean_tfidf,
})

# 두 기준의 순위를 각각 매긴다
term_table["Count순위"] = term_table["등장횟수"].rank(ascending=False, method="min").astype(int)
term_table["TFIDF순위"] = term_table["평균TFIDF"].rank(ascending=False, method="min").astype(int)
term_table["순위변동"] = term_table["Count순위"] - term_table["TFIDF순위"]

term_table.sort_values("Count순위").head(10)

,단어,등장횟수,DF,IDF,평균TFIDF,Count순위,TFIDF순위,순위변동
61,2026,60,60,3.783796,0.015419,1,2,-1
63,2027,56,56,3.851619,0.015500,2,1,1
1422,에디션,28,28,4.527374,0.009248,3,3,0
2092,해커스,25,25,4.636574,0.007563,4,4,0
452,기념,19,19,4.898938,0.005936,5,8,-3
457,기본서,18,18,4.950231,0.005839,6,9,-3
1183,세트,18,18,4.950231,0.006523,6,6,0
62,2026년,17,17,5.004298,0.006554,8,5,3
1948,토익,17,17,5.004298,0.006126,8,7,1
271,the,15,15,5.122081,0.005759,10,10,0


## 프롬프트 예시

```
단어별 지표가 든 DataFrame term_table이 있습니다.
Count 상위 60개 중에서 TF-IDF 순위가 가장 많이 내려간 단어 5개를 보고 싶습니다.
단어, 등장횟수, DF, 두 순위, IDF를 표로 보여 주세요.
함수로 만들지 말고 바로 실행되는 코드로 작성해 주세요.
try/except 예외 처리는 넣지 마세요.
5줄 이내로 짧게 작성해 주세요.
설명은 코드 옆 짧은 주석으로만 달아 주세요.
```

In [39]:
# 2) Count 상위 60개 중 TF-IDF에서 순위가 많이 내려간 단어
top60 = term_table.nsmallest(60, "Count순위")

print("=== 순위가 내려간 단어 (TF-IDF에서 덜 중요해짐) ===")
print(top60.nsmallest(5, "순위변동")[
    ["단어", "등장횟수", "DF", "Count순위", "TFIDF순위", "IDF"]
].to_string(index=False))

=== 순위가 내려간 단어 (TF-IDF에서 덜 중요해짐) ===


    단어  등장횟수  DF  Count순위  TFIDF순위      IDF
   공기업     6   6       53      166 5.948760
    독끝     8   8       33       96 5.697445
   권으로     6   6       53       88 5.948760
    통합     8   8       33       60 5.697445
해커스공무원     9   9       26       51 5.592085


## 프롬프트 예시

```
같은 DataFrame에서 반대로 TF-IDF 순위가
가장 많이 올라간 단어 5개를 보고 싶습니다.
함수로 만들지 말고 바로 실행되는 코드로 작성해 주세요.
try/except 예외 처리는 넣지 마세요.
5줄 이내로 짧게 작성해 주세요.
설명은 코드 옆 짧은 주석으로만 달아 주세요.
```

In [40]:
# 3) 반대로 순위가 올라간 단어
print("=== 순위가 올라간 단어 (TF-IDF에서 더 중요해짐) ===")
print(top60.nlargest(5, "순위변동")[
    ["단어", "등장횟수", "DF", "Count순위", "TFIDF순위", "IDF"]
].to_string(index=False))

=== 순위가 올라간 단어 (TF-IDF에서 더 중요해짐) ===
  단어  등장횟수  DF  Count순위  TFIDF순위      IDF
  부의     6   6       53       31 5.948760
흔한남매     7   7       46       25 5.815228
  나의     8   8       33       21 5.697445
  법칙     6   6       53       42 5.948760
 대모험     7   7       46       37 5.815228


### 여기서 이상한 점이 보인다

순위가 내려간 `공기업`(DF 6)과 올라간 `부의`(DF 6)는 **DF가 똑같다.**
IDF도 같다. 그런데 TF-IDF 순위는 크게 갈렸다.

IDF만으로는 설명이 안 된다. 다른 이유가 있다.

## 프롬프트 예시

```
DF가 같은데도 TF-IDF 순위가 갈린 단어가 있습니다.
'공기업'과 '부의' 두 단어에 대해
그 단어가 들어 있는 제목이 몇 개인지, 그 제목들의 평균 토큰 수가 몇 개인지,
그리고 실제 제목 3개를 출력해 원인을 확인하고 싶습니다.
함수로 만들지 말고 바로 실행되는 코드로 작성해 주세요.
try/except 예외 처리는 넣지 마세요.
5줄 이내로 짧게 작성해 주세요.
설명은 코드 옆 짧은 주석으로만 달아 주세요.
```

In [41]:
# 4) 두 단어가 들어 있는 제목의 길이를 비교해 본다
for term in ["공기업", "부의"]:
    col = count_vectorizer.vocabulary_[term]
    rows_with = np.asarray((X_count[:, col] > 0).todense()).ravel()
    lengths = np.asarray(X_count[rows_with].sum(axis=1)).ravel()

    print(f"[{term}] 등장 제목 {int(rows_with.sum())}개 / 제목당 평균 토큰 수 {lengths.mean():.1f}개")
    for t in titles[rows_with][:3]:
        print("    -", t)
    print()

[공기업] 등장 제목 6개 / 제목당 평균 토큰 수 10.7개
    - 2026 독끝 금융논술 주요 13대 기업 과년도 기출+예상문제: 공기업·은행권 필기/논술시험 대비
    - 2026 위포트 공기업 NCS 통합 기본서
    - 독끝 NCS 문제해결능력 자원관리능력 460제: 공기업·공사·공단 채용시험 대비 독학으로 끝내는 PSAT 세트

[부의] 등장 제목 6개 / 제목당 평균 토큰 수 2.7개
    - 부의 갈림길
    - 부의 인문학(20만부 기념 개정증보판)
    - 대한민국 부의 감각



### 원인은 제목의 길이다

`TfidfVectorizer`는 기본적으로 각 문서 벡터를 **길이 1로 정규화(L2)** 한다.
그래서 **한 제목 안에 단어가 적을수록 그 단어 하나가 갖는 값이 커진다.**

```
짧은 제목의 단어  ->  나눠 가질 단어가 적다  ->  TF-IDF 값이 커진다
긴 제목의 단어    ->  여러 단어가 나눠 갖는다 ->  TF-IDF 값이 작아진다
```

`부의`는 `부의 추월차선` 같은 짧은 제목에 들어 있고,
`공기업`은 `해커스공기업 NCS 기출 모의고사` 같은 긴 제목에 들어 있다.

> **TF-IDF 값은 IDF만으로 정해지지 않는다. 문서의 길이도 함께 작용한다.**
> 실습 19에서 "최종 TF-IDF를 단순한 등장 횟수로 해석하면 안 된다"고 한 이유다.

## 프롬프트 예시

```
단어별 지표가 든 DataFrame term_table이 있습니다.
기본 CountVectorizer가 숫자를 그대로 단어로 만드는지 확인하고 싶습니다.
숫자로만 이루어진 단어가 몇 개이고 전체의 몇 퍼센트인지,
그 단어들의 등장 횟수 합은 얼마인지,
Count 상위 10개 중 숫자 단어는 무엇인지 출력해 주세요.
함수로 만들지 말고 바로 실행되는 코드로 작성해 주세요.
try/except 예외 처리는 넣지 마세요.
5줄 이내로 짧게 작성해 주세요.
설명은 코드 옆 짧은 주석으로만 달아 주세요.
```

In [42]:
# 5) 기본 토큰 기준이 숫자를 어떻게 다루는지 확인
is_number = term_table["단어"].str.fullmatch(r"\d+")

print("숫자로만 이루어진 단어:", int(is_number.sum()), f"개 ({is_number.mean() * 100:.1f}%)")
print("숫자 단어의 등장 횟수 합:", int(term_table.loc[is_number, "등장횟수"].sum()))
print()
print("Count 상위 10개 중 숫자 단어:")
print(term_table.nsmallest(10, "Count순위").loc[lambda x: x["단어"].str.fullmatch(r"\d+"),
                                                ["단어", "등장횟수", "DF"]].to_string(index=False))

숫자로만 이루어진 단어: 48 개 (2.2%)
숫자 단어의 등장 횟수 합: 188

Count 상위 10개 중 숫자 단어:
  단어  등장횟수  DF
2026    60  60
2027    56  56


## 프롬프트 예시

```
단어별 지표가 든 term_table과 Count 행렬 X_count가 있습니다.
단어가 얼마나 흩어져 있는지 보고 싶습니다.
전체 단어 수, DF가 1인 단어 수와 비율,
제목당 평균 토큰 수, 토큰이 0개인 제목 수, IDF 범위를 출력해 주세요.
함수로 만들지 말고 바로 실행되는 코드로 작성해 주세요.
try/except 예외 처리는 넣지 마세요.
5줄 이내로 짧게 작성해 주세요.
설명은 코드 옆 짧은 주석으로만 달아 주세요.
```

In [43]:
# 6) 행렬이 얼마나 희소한지, 단어가 얼마나 흩어져 있는지
print("전체 단어 수:", len(term_table))
print("1개 문서에만 등장한 단어:", int((term_table["DF"] == 1).sum()),
      f"({(term_table['DF'] == 1).mean() * 100:.1f}%)")
print()
print("제목당 평균 토큰 수:", round(X_count.sum() / X_count.shape[0], 2))
print("토큰이 0개인 제목:", int((np.asarray(X_count.sum(axis=1)).ravel() == 0).sum()))
print()
print("IDF 범위:", round(tfidf_vectorizer.idf_.min(), 3), "~", round(tfidf_vectorizer.idf_.max(), 3))

전체 단어 수: 2179
1개 문서에만 등장한 단어: 1589 (72.9%)

제목당 평균 토큰 수: 3.82
토큰이 0개인 제목: 3

IDF 범위: 3.784 ~ 7.202


## 프롬프트 예시

```
단어별 지표가 든 term_table이 있습니다.
DF가 커지면 IDF가 작아지는지 직접 확인하고 싶습니다.
DF가 가장 큰 단어 5개와 DF가 1인 단어 5개를
각각 DF, IDF와 함께 출력해 주세요.
함수로 만들지 말고 바로 실행되는 코드로 작성해 주세요.
try/except 예외 처리는 넣지 마세요.
5줄 이내로 짧게 작성해 주세요.
설명은 코드 옆 짧은 주석으로만 달아 주세요.
```

In [44]:
# 7) DF가 커지면 IDF가 작아지는지 직접 확인
print("DF가 가장 큰 단어 5개 (흔한 단어):")
print(term_table.nlargest(5, "DF")[["단어", "DF", "IDF"]].to_string(index=False))
print()
print("DF가 1인 단어 예시 5개 (드문 단어):")
print(term_table[term_table["DF"] == 1].head(5)[["단어", "DF", "IDF"]].to_string(index=False))

DF가 가장 큰 단어 5개 (흔한 단어):
  단어  DF      IDF
2026  60 3.783796
2027  56 3.851619
 에디션  28 4.527374
 해커스  25 4.636574
  기념  19 4.898938

DF가 1인 단어 예시 5개 (드문 단어):
   단어  DF      IDF
  007   1 7.201523
 100만   1 7.201523
100만부   1 7.201523
 100일   1 7.201523
100주년   1 7.201523


### 확인한 것

`2026`은 60개 제목에 등장해 IDF가 가장 낮은 **3.784**,
1개 제목에만 등장한 단어들은 IDF가 가장 높은 **7.202**였다.

DF가 커질수록 IDF가 작아진다는 것이 실제 숫자로 확인된다.
다만 이번 데이터는 **전체 단어의 72.9%가 DF 1**이라
IDF 값 대부분이 최댓값 근처에 몰려 있다.

> 도서 제목처럼 **짧고 겹치지 않는 문서**에서는
> IDF가 단어를 구분하는 힘이 생각보다 약할 수 있다.

---
# 실습 33. CountVectorizer와 TfidfVectorizer의 역할 비교

| 구분 | CountVectorizer | TfidfVectorizer |
|---|---|---|
| 기본 값 | 단어 등장 횟수 | TF-IDF 가중치 |
| 값 형태 | 주로 정수 | 실수 |
| 전체 문서의 흔함 고려 | 직접 고려하지 않음 | IDF로 고려 |
| 장점 | 단순하고 직관적 | 흔한 단어의 영향 일부 조정 |
| 주요 활용 | 빈도 기반 특징 | 분류·유사도 등 텍스트 특징 |

**둘 중 하나가 항상 더 좋다고 단정하지 않는다.**
분석 목적과 모델, 데이터에 따라 성능을 실제로 검증해야 한다.

---
# 실습 34. 같은 단어 사전을 사용하는지 확인하기

기본 설정이 같더라도 두 Vectorizer를 별도로 `fit()`하면
일반적으로 비슷한 단어 사전을 만든다.
하지만 정확한 비교를 위해 **직접 확인하는 것이 안전하다.**

## 프롬프트 예시

```
CountVectorizer와 TfidfVectorizer를 각각 따로 fit했습니다.
두 Vectorizer가 만든 단어 집합이 정말 같은지 확인하고 싶습니다.
각각의 단어 수와 두 집합이 같은지 여부를 출력해 주세요.
함수로 만들지 말고 바로 실행되는 코드로 작성해 주세요.
try/except 예외 처리는 넣지 마세요.
5줄 이내로 짧게 작성해 주세요.
설명은 코드 옆 짧은 주석으로만 달아 주세요.
```

In [45]:
count_feature_set = set(count_vectorizer.get_feature_names_out())
tfidf_feature_set = set(tfidf_vectorizer.get_feature_names_out())

print("Count 단어 수:", len(count_feature_set))
print("TF-IDF 단어 수:", len(tfidf_feature_set))
print("같은 단어 집합인가?:", count_feature_set == tfidf_feature_set)

Count 단어 수: 2179
TF-IDF 단어 수: 2179
같은 단어 집합인가?: True


---
# 실습 35. stop_words 옵션 이해하기

## 해야 할 일

Vectorizer에서도 불용어를 지정할 수 있다.
하지만 **불용어는 무조건 많이 제거하는 것이 좋은 것이 아니다.**

### 잘못된 접근
> AI가 추천한 불용어 목록 200개를 검토 없이 그대로 사용

### 권장 접근
> 상위 단어 확인 -> 실제 제목에서 사용 방식 확인 -> 분석에 의미가 적은지 판단
> -> 필요한 단어만 불용어로 추가 -> 다시 결과 확인

## 프롬프트 예시

```
TfidfVectorizer에 불용어를 지정해 보고 싶습니다.
'그리고', '대한', '위한' 세 단어를 불용어로 지정해 titles에 적용하고,
불용어 적용 전후의 단어 수가 몇 개나 달라졌는지 비교해 주세요.
함수로 만들지 말고 바로 실행되는 코드로 작성해 주세요.
try/except 예외 처리는 넣지 마세요.
5줄 이내로 짧게 작성해 주세요.
설명은 코드 옆 짧은 주석으로만 달아 주세요.
```

In [46]:
stop_words = [
    "그리고",
    "대한",
    "위한",
]

vectorizer_with_stopwords = TfidfVectorizer(stop_words=stop_words)
X_stop = vectorizer_with_stopwords.fit_transform(titles)

print("불용어 적용 전 단어 수:", X_tfidf.shape[1])
print("불용어 적용 후 단어 수:", X_stop.shape[1])
print("줄어든 단어 수:", X_tfidf.shape[1] - X_stop.shape[1])

불용어 적용 전 단어 수: 2179
불용어 적용 후 단어 수: 2177
줄어든 단어 수: 2


---
# 실습 36. max_features 옵션 이해하기

단어 수가 너무 많을 때 **최대 feature 수를 제한**할 수 있다.

하지만 이번 실습에서 무조건 1000을 정답처럼 사용하지 않는다.
먼저 기본 설정으로 전체 단어 수를 확인하고, **왜 제한이 필요한지 이해한 뒤** 사용한다.

## 프롬프트 예시

```
TfidfVectorizer의 max_features 옵션을 확인하고 싶습니다.
현재 단어 수를 먼저 출력하고,
max_features=1000을 적용했을 때 단어 수가 어떻게 달라지는지 비교해 주세요.
함수로 만들지 말고 바로 실행되는 코드로 작성해 주세요.
try/except 예외 처리는 넣지 마세요.
5줄 이내로 짧게 작성해 주세요.
설명은 코드 옆 짧은 주석으로만 달아 주세요.
```

In [47]:
print("현재 단어 수:", len(tfidf_terms))

limited_vectorizer = TfidfVectorizer(max_features=1000)
X_limited = limited_vectorizer.fit_transform(titles)

print("max_features=1000 적용 후:", X_limited.shape[1])

현재 단어 수: 2179
max_features=1000 적용 후: 1000


---
# 실습 37. min_df 옵션 이해하기

`min_df`는 **너무 적은 문서에서만 등장하는 단어를 제외**할 때 사용한다.

`min_df=2`는 대략 다음 의미다.

> 한 문서에만 등장한 단어는 제외하고, 2개 이상의 문서에 등장한 단어 사용

하지만 **드문 단어가 항상 쓸모없는 것은 아니다.**
도서 제목에서는 특정 전문용어가 카테고리를 구분하는 중요한 특징일 수도 있다.
옵션을 적용하면 반드시 **전후 feature 수와 모델 결과를 비교**해야 한다.

## 프롬프트 예시

```
TfidfVectorizer의 min_df 옵션을 확인하고 싶습니다.
min_df=2를 적용해 titles를 변환하고,
적용 전후의 단어 수와 제외된 단어 수를 비교해 주세요.
함수로 만들지 말고 바로 실행되는 코드로 작성해 주세요.
try/except 예외 처리는 넣지 마세요.
5줄 이내로 짧게 작성해 주세요.
설명은 코드 옆 짧은 주석으로만 달아 주세요.
```

In [48]:
vectorizer_min_df = TfidfVectorizer(min_df=2)
X_min_df = vectorizer_min_df.fit_transform(titles)

print("min_df 적용 전 단어 수:", X_tfidf.shape[1])
print("min_df=2 적용 후 단어 수:", X_min_df.shape[1])
print("제외된 단어 수:", X_tfidf.shape[1] - X_min_df.shape[1])
print()
print("-> 제외된 단어는 모두 '1개 문서에만 등장한 단어'다.")

min_df 적용 전 단어 수: 2179
min_df=2 적용 후 단어 수: 590
제외된 단어 수: 1589

-> 제외된 단어는 모두 '1개 문서에만 등장한 단어'다.


---
# 실습 38. max_df 옵션 이해하기

`max_df`는 **지나치게 많은 문서에 등장하는 단어**를 자동으로 제외할 때 사용한다.

`max_df=0.95`는 95%보다 많은 문서에 등장하는 단어를 제외하는 설정이다.
이번 데이터에서는 실제로 그런 단어가 거의 없을 수도 있다.

> **옵션을 넣었다는 사실보다 왜 넣었고 결과가 어떻게 바뀌었는지가 중요하다.**

## 프롬프트 예시

```
TfidfVectorizer의 max_df 옵션을 확인하고 싶습니다.
max_df=0.95를 적용해 titles를 변환하고,
적용 전후의 단어 수와 제외된 단어 수를 비교해 주세요.
함수로 만들지 말고 바로 실행되는 코드로 작성해 주세요.
try/except 예외 처리는 넣지 마세요.
5줄 이내로 짧게 작성해 주세요.
설명은 코드 옆 짧은 주석으로만 달아 주세요.
```

In [49]:
vectorizer_max_df = TfidfVectorizer(max_df=0.95)
X_max_df = vectorizer_max_df.fit_transform(titles)

print("max_df 적용 전 단어 수:", X_tfidf.shape[1])
print("max_df=0.95 적용 후 단어 수:", X_max_df.shape[1])
print("제외된 단어 수:", X_tfidf.shape[1] - X_max_df.shape[1])

max_df 적용 전 단어 수: 2179
max_df=0.95 적용 후 단어 수: 2179
제외된 단어 수: 0


---
# 실습 39. ngram_range 개념 맛보기

기본 Vectorizer는 보통 **한 단어씩** feature를 만든다. 이를 **unigram**이라고 한다.

```
데이터 분석 입문  ->  데이터 / 분석 / 입문
```

두 단어 조합(**bigram**)도 함께 사용하도록 설정할 수 있다.

> `ngram_range`는 확장 개념이다. 기본 unigram을 먼저 이해한 뒤 필요할 때 실험한다.

## 프롬프트 예시

```
TfidfVectorizer의 ngram_range 옵션을 확인하고 싶습니다.
문장 3개가 든 sample_docs에 ngram_range=(1, 2)를 적용하고
만들어진 feature 목록을 출력해 주세요.
두 단어 조합이 함께 들어가는지 보고 싶습니다.
함수로 만들지 말고 바로 실행되는 코드로 작성해 주세요.
try/except 예외 처리는 넣지 마세요.
5줄 이내로 짧게 작성해 주세요.
설명은 코드 옆 짧은 주석으로만 달아 주세요.
```

In [50]:
bigram_vectorizer = TfidfVectorizer(ngram_range=(1, 2))
X_bigram = bigram_vectorizer.fit_transform(sample_docs)

print(bigram_vectorizer.get_feature_names_out())

['데이터' '데이터 분석' '머신러닝' '분석' '분석 입문' '입문' '파이썬' '파이썬 데이터' '파이썬 머신러닝']


---
# 실습 40. Chapter 02 형태소 분석 결과와 연결하기

## 해야 할 일

Chapter 02에서는 Kiwi로 명사 중심 단어를 추출했다.
같은 전처리 철학을 Vectorizer에도 적용할 수 있다.

> 각 제목을 형태소 분석해서 **사용할 단어만 문자열로 다시 연결**한 뒤 Vectorizer에 넣는다.

### 중요한 점

이 방식이 무조건 기본 Vectorizer보다 좋다고 단정하지 않는다.
**기본 제목 문자열 vs Kiwi 전처리 제목** 두 방식의 feature와
이후 모델 성능을 비교해서 판단해야 한다.

## 프롬프트 예시

```
Chapter 02에서 쓴 Kiwi 형태소 분석을 Vectorizer와 연결하려고 합니다.
문장을 넣으면 명사(NNG, NNP)와 영문(SL)만 골라 리스트로 돌려주는
extract_terms(text) 함수를 만들고 싶습니다.
2글자 미만 단어와 '도서', '책'은 제외해 주세요.
try/except 예외 처리는 넣지 마세요.
초보자가 읽기 쉽게 10줄 이내로 짧게 작성해 주세요.
설명은 코드 옆 짧은 주석으로만 달아 주세요.
```

In [51]:
from kiwipiepy import Kiwi

kiwi = Kiwi()
USE_TAGS = {"NNG", "NNP", "SL"}
STOP_WORDS = {"도서", "책"}

def extract_terms(text):
    terms = []
    for token in kiwi.tokenize(str(text)):
        word = token.form.strip()
        if token.tag not in USE_TAGS:
            continue
        if len(word) < 2:
            continue
        if word in STOP_WORDS:
            continue
        terms.append(word)
    return terms

print(extract_terms("AI 시대의 데이터 분석을 위한 파이썬"))

['AI', '시대', '데이터', '분석', '파이썬']


## 프롬프트 예시

```
extract_terms(text) 함수와 제목 Series titles가 있습니다.
각 제목을 형태소 분석한 결과를 공백으로 이어 붙여
processed_titles라는 Series로 만들고 앞의 5개를 확인하고 싶습니다.
함수로 만들지 말고 바로 실행되는 코드로 작성해 주세요.
try/except 예외 처리는 넣지 마세요.
5줄 이내로 짧게 작성해 주세요.
설명은 코드 옆 짧은 주석으로만 달아 주세요.
```

In [52]:
# 각 제목을 공백으로 다시 연결한다
processed_titles = titles.apply(
    lambda text: " ".join(extract_terms(text))
)
processed_titles.head()

0               세네카 오늘
1                   남매
2               머니 트렌드
3                 싯다르타
4    한국사 이상 현상 연구원 일반판
Name: 상품명, dtype: str

## 프롬프트 예시

```
Kiwi로 전처리한 제목 processed_titles가 있습니다.
여기에 TfidfVectorizer를 적용하고,
원래 제목으로 만든 X_tfidf와 행렬 크기를 비교해 주세요.
함수로 만들지 말고 바로 실행되는 코드로 작성해 주세요.
try/except 예외 처리는 넣지 마세요.
5줄 이내로 짧게 작성해 주세요.
설명은 코드 옆 짧은 주석으로만 달아 주세요.
```

In [53]:
kiwi_tfidf_vectorizer = TfidfVectorizer()
X_kiwi_tfidf = kiwi_tfidf_vectorizer.fit_transform(processed_titles)

print("기본 제목 TF-IDF :", X_tfidf.shape)
print("Kiwi 전처리 TF-IDF:", X_kiwi_tfidf.shape)

기본 제목 TF-IDF : (986, 2179)
Kiwi 전처리 TF-IDF: (986, 1396)


---
# 실습 41. Custom tokenizer 방식은 언제 사용할까?

Vectorizer에 직접 tokenizer 함수를 전달할 수도 있다.

하지만 초보자 과정에서는 **전처리된 문자열을 먼저 만들고 그 결과를 Vectorizer에 넣는 방식**이
중간 결과를 확인하기 더 쉽다.

```
원본 제목 -> Kiwi 결과 확인 -> 전처리 제목 확인 -> Vectorizer 적용
```

이렇게 단계별로 확인할 수 있기 때문이다.

## 프롬프트 예시

```
TfidfVectorizer에 직접 tokenizer 함수를 넘기는 방식을 확인하고 싶습니다.
extract_terms를 쓰는 kiwi_tokenizer 함수를 만들어
TfidfVectorizer(tokenizer=..., token_pattern=None, lowercase=False)로 적용하고,
전처리 문자열을 먼저 만드는 방식과 결과 크기를 비교해 주세요.
함수로 만들지 말고 바로 실행되는 코드로 작성해 주세요.
try/except 예외 처리는 넣지 마세요.
5줄 이내로 짧게 작성해 주세요.
설명은 코드 옆 짧은 주석으로만 달아 주세요.
```

In [54]:
def kiwi_tokenizer(text):
    return extract_terms(text)

custom_vectorizer = TfidfVectorizer(
    tokenizer=kiwi_tokenizer,
    token_pattern=None,
    lowercase=False,
)
X_custom = custom_vectorizer.fit_transform(titles)

print("custom tokenizer 결과:", X_custom.shape)
print("전처리 문자열 방식 결과:", X_kiwi_tfidf.shape)

custom tokenizer 결과: (986, 1377)
전처리 문자열 방식 결과: (986, 1396)


---
# 실습 42. Vectorizer가 만든 결과를 시각적으로 이해하기

작은 예제에서 다음 표를 떠올린다.

```
문서                  데이터   머신러닝   분석   입문   파이썬
파이썬 데이터 분석       1        0        1     0      1
파이썬 머신러닝          0        1        0     0      1
데이터 분석 입문         1        0        1     1      0
```

이 표의 **한 행은 하나의 문서를 숫자로 표현한 벡터**다.

```
문서 1 -> [1, 0, 1, 0, 1]
```

TF-IDF에서도 구조는 같다. 다만 셀 값이 단순 Count가 아니라 **가중치**가 된다.

```
문서 1 -> [0.52, 0.00, 0.52, 0.00, 0.40]
```

---
# 실습 43. 행렬에서 0이 많다는 의미

각 도서 제목은 **짧다.**
전체 데이터에는 수많은 단어가 있지만 한 제목에는 그중 일부만 등장한다.

```
도서 A -> [0, 0, 0.7, 0, 0.4, 0, 0, ...]
도서 B -> [0.5, 0, 0, 0, 0, 0.6, 0, ...]
```

이것이 **희소 행렬을 사용하는 이유**와 연결된다.
Chapter 05의 코사인 유사도에서도 이 TF-IDF 벡터를 이용하게 된다.

---
# 실습 44. 0이 아닌 값의 개수 확인하기

## 프롬프트 예시

```
TF-IDF 행렬 X_tfidf가 왜 희소한지 숫자로 확인하고 싶습니다.
전체 셀 수, 0이 아닌 셀 수, 그리고 0의 비율을 계산해 출력해 주세요.
0의 비율은 소수점 4자리까지만 보여 주세요.
함수로 만들지 말고 바로 실행되는 코드로 작성해 주세요.
try/except 예외 처리는 넣지 마세요.
5줄 이내로 짧게 작성해 주세요.
설명은 코드 옆 짧은 주석으로만 달아 주세요.
```

In [55]:
rows, cols = X_tfidf.shape
total_cells = rows * cols
non_zero_cells = X_tfidf.nnz

print("전체 셀 수:", total_cells)
print("0이 아닌 셀 수:", non_zero_cells)

zero_ratio = 1 - (non_zero_cells / total_cells)
print("0의 비율:", round(zero_ratio, 4))

전체 셀 수: 2148494
0이 아닌 셀 수: 3744
0의 비율: 0.9983


이 결과로 텍스트 행렬이 왜 **sparse(희소)** 한지 직접 확인할 수 있다.

---
# 실습 45. AI에게 결과 해석을 요청할 때 주의하기

인공지능에게 **숫자 없이** 이렇게 질문하지 않는다.

```
TF-IDF 결과를 분석해 주세요.
```

대신 실제 결과를 함께 전달한다.

## 프롬프트 예시

```
교보문고 베스트셀러 도서 제목을 TfidfVectorizer로 변환했습니다.
실제 결과는 다음과 같습니다.
문서 수: [실제 값]
단어 수: [실제 값]
평균 TF-IDF 상위 10개:
[실제 출력 붙여넣기]
다음 조건으로 설명해 주세요.
1. 숫자를 임의로 만들지 말 것
2. TF-IDF가 높은 단어를 판매 원인으로 단정하지 말 것
3. 현재 문서 집합에서 상대적으로 두드러진 텍스트 특징이라는 수준으로 설명할 것
4. 5문장 이내로 작성할 것
```

**AI가 작성한 설명도 실제 출력과 다시 비교한다.**

---
# 실습 46. Count와 TF-IDF 차이를 Markdown으로 정리하기

> 아래는 위 셀들의 **실제 실행 결과**를 보고 적은 것이다.

## CountVectorizer와 TF-IDF 비교

CountVectorizer는 각 도서 제목에서 단어가 등장한 **횟수**를 숫자로 표현했다.
반면 TF-IDF는 한 제목에서의 등장뿐 아니라
전체 제목에서 그 단어가 얼마나 흔하게 등장하는지도 반영했다.

실제 결과에서 두 상위 목록은 다음과 같았다.

| 순위 | Count 상위 | 전체등장횟수 | TF-IDF 상위 | 평균 TF-IDF |
|---|---|---|---|---|
| 1 | 2026 | 60 | 2027 | 0.01550 |
| 2 | 2027 | 56 | 2026 | 0.01542 |
| 3 | 에디션 | 28 | 에디션 | 0.00925 |
| 4 | 해커스 | 25 | 해커스 | 0.00756 |
| 5 | 기념 | 19 | 2026년 | 0.00655 |

상위권 단어 자체는 크게 다르지 않았지만 **순위가 일부 바뀌었다.**
`2026`은 Count 기준 1위(60회)였지만 평균 TF-IDF에서는 `2027`에 1위를 내주었다.
`2026`이 더 많은 제목에 퍼져 있어(DF 60 vs 56) IDF가 낮아졌기 때문이다.

## 순위가 크게 움직인 단어

상위 60개로 범위를 넓혀 보면 변화가 더 뚜렷하다.

| 방향 | 단어 | 등장 | DF | Count순위 | TFIDF순위 |
|---|---|---|---|---|---|
| 내려감 | 공기업 | 6 | 6 | 53위 | 166위 |
| 내려감 | 독끝 | 8 | 8 | 33위 | 96위 |
| 내려감 | 해커스공무원 | 9 | 9 | 26위 | 51위 |
| 올라감 | 부의 | 6 | 6 | 53위 | 31위 |
| 올라감 | 흔한남매 | 7 | 7 | 46위 | 25위 |
| 올라감 | 나의 | 8 | 8 | 33위 | 21위 |

여기서 중요한 점이 하나 있다.
`공기업`과 `부의`는 **DF가 6으로 똑같고 IDF도 같은데** 순위가 정반대로 갈렸다.

원인은 **제목의 길이**였다.
`TfidfVectorizer`는 각 문서 벡터를 길이 1로 정규화(L2)하기 때문에,
**한 제목에 단어가 적을수록 그 단어 하나가 갖는 값이 커진다.**

```
부의    -> '부의 추월차선'처럼 짧은 제목        -> 값이 커짐
공기업  -> '해커스공기업 NCS 기출...' 긴 제목   -> 여러 단어가 나눠 가짐
```

> **TF-IDF는 IDF만으로 정해지지 않는다. 문서의 길이도 함께 작용한다.**

## 이 데이터에서 IDF는 생각보다 힘이 약했다

- 전체 단어 2,179개 중 **1,589개(72.9%)가 DF 1**, 즉 한 제목에만 등장했다.
- 그래서 IDF 값 대부분이 최댓값(7.202) 근처에 몰려 있다.
- IDF 범위는 3.784(`2026`, DF 60) ~ 7.202(DF 1)로 두 배가 채 안 된다.
- 제목당 평균 토큰은 **3.82개**로 매우 짧다.

문서가 길고 단어가 여러 문서에 반복되는 뉴스나 논문에서는 IDF가 잘 작동한다.
하지만 **도서 제목처럼 짧고 겹치지 않는 문서**에서는
대부분의 단어가 비슷한 IDF를 갖게 되어 구분하는 힘이 떨어진다.

## 기본 토큰 기준이 숫자를 그대로 단어로 취급한다

Count 상위 1·2위가 `2026`, `2027`이었다.

- 숫자로만 된 단어는 48개(전체의 2.2%)뿐이지만, 등장 횟수는 **188회**다.
- 이 숫자들은 `2027 해커스경찰...`, `머니 트렌드 2027`처럼 **연도 표기**다.

Chapter 02에서 Kiwi로 뽑았을 때 상위가 `해커스`, `기출`, `기념`이었던 것과 다르다.
**어느 쪽이 틀린 것이 아니라 전처리 조건이 다른 것이다.**

따라서 **단순 등장 횟수와 문서 구분에 사용할 상대적 가중치는 같은 개념이 아니다.**

---
# 실습 47. 일부 도서 결과를 직접 검증하기

## 검증 질문

- 상위 단어가 실제 제목에 존재하는가?
- 숫자나 기호가 이상하게 feature가 되지는 않았는가?
- 너무 일반적인 단어가 높은 값으로 나오지는 않았는가?
- 형태소 분석을 적용하면 더 나아질 여지가 있는가?

## 프롬프트 예시

```
show_top_tfidf_terms(doc_index, top_n) 함수가 있습니다.
0번, 10번, 20번 도서의 TF-IDF 상위 5개 단어를 차례로 확인하고 싶습니다.
도서마다 구분선을 넣고 표 형태가 유지되도록 display()를 사용해 주세요.
함수로 만들지 말고 바로 실행되는 코드로 작성해 주세요.
try/except 예외 처리는 넣지 마세요.
5줄 이내로 짧게 작성해 주세요.
설명은 코드 옆 짧은 주석으로만 달아 주세요.
```

In [56]:
sample_indices = [0, 10, 20]

for index in sample_indices:
    if index < len(titles):
        print("=" * 60)
        display(show_top_tfidf_terms(index, top_n=5))

도서 제목: 세네카, 오늘을 빼앗기고 있는 당신에게


,단어,TF-IDF
0,빼앗기고,0.479241
1,세네카,0.452258
2,오늘을,0.452258
3,당신에게,0.452258
4,있는,0.395873


도서 제목: 수족관


,단어,TF-IDF
0,수족관,1.0


도서 제목: 브람스를 좋아하세요


,단어,TF-IDF
0,브람스를,0.707107
1,좋아하세요,0.707107


---
# 실습 48. 결과가 이상할 때 확인할 순서

TF-IDF 결과가 예상과 달라도 **바로 코드를 전부 바꾸지 않는다.**

1. 원본 제목 확인
2. 결측치 처리 확인
3. Vectorizer feature 확인
4. 토큰화 기준 확인
5. 불용어 확인
6. 최소 문서 빈도 옵션 확인
7. 형태소 분석 적용 여부 확인
8. 해당 행의 0이 아닌 값 확인

오류나 이상한 결과를 AI에게 질문할 때도 **이 정보를 함께 전달한다.**

---
# 실습 49. 학습용 행렬 저장하기

## 매우 중요한 주의

이 파일은 이번 Chapter에서 **벡터화 결과를 학습하고 확인하기 위한 결과물**이다.

다음 Chapter의 분류 모델 학습에서는
전체 데이터로 미리 `fit()`한 TF-IDF 결과를 **그대로 가져다 쓰지 않는다.**
분류에서는 학습 데이터와 테스트 데이터를 먼저 나눈 뒤 올바른 순서로 다시 Vectorizer를 학습한다.

## 프롬프트 예시

```
Count 행렬 X_count와 TF-IDF 행렬 X_tfidf를 파일로 저장하고 싶습니다.
scipy의 save_npz를 이용해
chapter03_count_matrix.npz와 chapter03_tfidf_matrix.npz로 저장하고,
파일이 만들어졌는지 확인하는 코드도 넣어 주세요.
함수로 만들지 말고 바로 실행되는 코드로 작성해 주세요.
try/except 예외 처리는 넣지 마세요.
5줄 이내로 짧게 작성해 주세요.
설명은 코드 옆 짧은 주석으로만 달아 주세요.
```

In [57]:
from scipy.sparse import save_npz

save_npz("chapter03_count_matrix.npz", X_count)
save_npz("chapter03_tfidf_matrix.npz", X_tfidf)

print("count 행렬 저장:", Path("chapter03_count_matrix.npz").exists())
print("tfidf 행렬 저장:", Path("chapter03_tfidf_matrix.npz").exists())

count 행렬 저장: True
tfidf 행렬 저장: True


---
# 실습 50. 왜 Chapter 04에서는 전체 데이터에 먼저 fit하면 안 될까?

이번 Chapter에서는 Vectorizer 자체를 이해하기 위해 전체 제목을 사용했다.

```
전체 도서 제목 -> fit_transform() -> 단어 사전과 TF-IDF 확인
```

이것은 **탐색과 개념 학습 목적**이다.

하지만 다음 Chapter에서는 **모델 성능을 평가**한다.
모델 평가에서 테스트 데이터는 **학습 단계에서 보지 않은 데이터**여야 한다.

만약 다음처럼 전체 데이터에 먼저 TF-IDF를 `fit()`하면 문제가 생긴다.

```
전체 데이터 -> TfidfVectorizer.fit_transform() -> train/test 분리
```

이 경우 Vectorizer가 단어 사전과 IDF를 만들 때
**이미 테스트 데이터의 정보를 본 상태**가 된다.
이를 **데이터 누수(Data Leakage)** 문제로 볼 수 있다.

---
# 실습 51. Chapter 04에서 사용할 올바른 순서 미리 보기

```
원본 데이터
    ↓
X와 y 준비
    ↓
train / test 분리
    ↓
TF-IDF를 train에 fit
    ↓
train transform
    ↓
test는 transform만 수행
    ↓
Naive Bayes 학습
    ↓
test 예측 및 평가
```

Python 흐름은 다음과 비슷하다.

```python
# Chapter 04에서 다룰 흐름의 개념 예시
X_train_tfidf = vectorizer.fit_transform(X_train)
X_test_tfidf = vectorizer.transform(X_test)
```

**테스트 데이터에는 `fit_transform()`을 사용하지 않는 것이 핵심이다.**

```python
# 사용하지 않을 방식
X_test_tfidf = vectorizer.fit_transform(X_test)
```

---
# 실습 52. fit과 transform을 구분해서 설명해 보기

이번 Chapter를 마치기 전에 다음 문장을 **본인의 말로** 설명해 본다.

| 메서드 | 하는 일 |
|---|---|
| `fit` | 데이터에서 단어 사전과 필요한 통계 정보를 **학습** |
| `transform` | 이미 학습한 기준을 이용해 데이터를 **숫자로 변환** |
| `fit_transform` | fit과 transform을 **한 번에** 수행 |

CountVectorizer와 TfidfVectorizer 모두 이 구분이 중요하다.
특히 머신러닝 평가에서는 **어떤 데이터에 fit했는가**가 매우 중요하다.

## 프롬프트 예시

```
이미 fit이 끝난 tfidf_vectorizer가 있습니다.
fit과 transform의 차이를 직접 보고 싶습니다.
새 제목 두 개를 transform()만 해서 변환하고,
열 수가 기존 단어 사전과 같은지, 각 제목에서 어떤 단어가 인식되는지 출력해 주세요.
학습 당시 없던 단어는 인식되지 않는다는 점을 확인하고 싶습니다.
함수로 만들지 말고 바로 실행되는 코드로 작성해 주세요.
try/except 예외 처리는 넣지 마세요.
5줄 이내로 짧게 작성해 주세요.
설명은 코드 옆 짧은 주석으로만 달아 주세요.
```

In [58]:
# fit과 transform의 차이를 직접 확인한다
new_titles = ["파이썬 데이터 분석 입문", "처음보는단어들로만된제목"]

# 이미 학습된 tfidf_vectorizer로 변환만 한다 (fit 하지 않음)
X_new = tfidf_vectorizer.transform(new_titles)

print("변환 결과 크기:", X_new.shape)
print("-> 열 수가 기존 단어 사전과 같다:", X_new.shape[1] == X_tfidf.shape[1])
print()

for i, title in enumerate(new_titles):
    row = X_new.getrow(i)
    print(f"[{title}]")
    print("  인식된 단어:", tfidf_terms[row.indices].tolist())

변환 결과 크기: (2, 2179)
-> 열 수가 기존 단어 사전과 같다: True

[파이썬 데이터 분석 입문]
  인식된 단어: []
[처음보는단어들로만된제목]
  인식된 단어: []


두 번째 제목은 **학습 당시 없던 단어**이므로 인식된 단어가 없다.
`transform`은 새 단어를 사전에 추가하지 않고 **학습된 기준만 사용**하기 때문이다.

---
# 실습 53. Vectorizer 옵션 기록하기

## Chapter 03 벡터화 조건

- 데이터: `book_bestseller_clean.csv`
- 분석 컬럼: `상품명`
- 사용한 제목 수: 986개
- CountVectorizer: 기본 설정
- TfidfVectorizer: 기본 설정
- 만들어진 단어 수: 2,179개 (두 Vectorizer 동일)
- 실습 목적: 전체 문서에서 벡터화 구조와 feature 확인
- 형태소 분석 비교: Kiwi(NNG/NNP/SL, 2글자 이상, 불용어 도서·책) 전처리 버전도 함께 생성
- 모델 평가용 Vectorizer: **Chapter 04에서 train 데이터 기준으로 별도 fit 예정**

---
# 실습 54. 재현 가능한 코드 구조로 정리하기

## 해야 할 일

Notebook 마지막에는 핵심 코드를 한 번 정리한다.

> 이 코드는 전체 흐름을 **다시 확인하기 위한 정리용**이다.
> 각 단계의 원리를 이해하지 않은 채 이 코드만 복사하는 것은 권장하지 않는다.

## 프롬프트 예시

```
pandas DataFrame df_books에 '상품명' 컬럼이 있습니다.
이 컬럼을 Vectorizer에 넣을 수 있도록 문자열로 정리하고 싶습니다.
1. 결측치는 빈 문자열로 바꾸고
2. 모두 문자열로 통일하고
3. 앞뒤 공백을 제거하고
4. 내용이 없는 제목은 제외하고
5. 인덱스를 0부터 다시 매겨서 titles라는 Series로 만들어 주세요.
마지막에 제목 개수와 앞의 10개를 확인하는 코드도 넣어 주세요.
함수로 만들지 말고 바로 실행되는 코드로 작성해 주세요.
try/except 예외 처리는 넣지 마세요.
5줄 이내로 짧게 작성해 주세요.
설명은 코드 옆 짧은 주석으로만 달아 주세요.
```

In [59]:
import numpy as np
import pandas as pd
from sklearn.feature_extraction.text import (
    CountVectorizer,
    TfidfVectorizer,
)

DATA_PATH = "book_bestseller_clean.csv"

# 1. 데이터 불러오기
df_books = pd.read_csv(DATA_PATH, encoding="utf-8-sig")

# 2. 제목 준비
titles = (
    df_books["상품명"]
    .fillna("")
    .astype(str)
    .str.strip()
)
titles = titles[titles != ""].reset_index(drop=True)

# 3. CountVectorizer
count_vectorizer = CountVectorizer()
X_count = count_vectorizer.fit_transform(titles)
count_terms = count_vectorizer.get_feature_names_out()

# 4. Count 전체 빈도
count_sums = np.asarray(X_count.sum(axis=0)).ravel()
count_summary = pd.DataFrame({
    "단어": count_terms,
    "전체등장횟수": count_sums,
}).sort_values(
    "전체등장횟수",
    ascending=False,
).reset_index(drop=True)

# 5. TF-IDF
tfidf_vectorizer = TfidfVectorizer()
X_tfidf = tfidf_vectorizer.fit_transform(titles)
tfidf_terms = tfidf_vectorizer.get_feature_names_out()

# 6. 평균 TF-IDF
mean_tfidf = np.asarray(X_tfidf.mean(axis=0)).ravel()
tfidf_summary = pd.DataFrame({
    "단어": tfidf_terms,
    "평균_TFIDF": mean_tfidf,
}).sort_values(
    "평균_TFIDF",
    ascending=False,
).reset_index(drop=True)

# 7. 결과 확인
print("Count shape:", X_count.shape)
print("TF-IDF shape:", X_tfidf.shape)
display(count_summary.head(30))
display(tfidf_summary.head(30))

Count shape: (986, 2179)
TF-IDF shape: (986, 2179)


,단어,전체등장횟수
0,2026,60
1,2027,56
2,에디션,28
3,해커스,25
4,기념,19
5,기본서,18
6,세트,18
7,2026년,17
8,토익,17
9,the,15


,단어,평균_TFIDF
0,2027,0.015500
1,2026,0.015419
2,에디션,0.009248
3,해커스,0.007563
4,2026년,0.006554
5,세트,0.006523
6,토익,0.006126
7,기념,0.005936
8,기본서,0.005839
9,the,0.005759


---
# 실습 55. Notebook 최종 실행 확인

> Kernel Restart -> Run All -> 마지막 셀까지 오류 없이 실행

## 확인 체크리스트

- [x] `book_bestseller_clean.csv`를 정상적으로 불러왔다.
- [x] 상품명 문자열을 정리했다.
- [x] 작은 예제로 Bag of Words를 설명할 수 있다.
- [x] CountVectorizer의 `fit_transform()`을 실행했다.
- [x] `get_feature_names_out()` 결과를 확인했다.
- [x] 행이 문서이고 열이 단어라는 것을 이해했다.
- [x] Count 행렬의 값이 단어 등장 횟수임을 이해했다.
- [x] 실제 도서 제목 한 행을 직접 검증했다.
- [x] sparse matrix를 전체 Dense 배열로 무조건 변환하지 않았다.
- [x] 전체 Count 상위 단어를 확인했다.
- [x] `chapter03_count_top_terms.csv`를 저장했다.
- [x] TF, DF, IDF의 역할을 구분할 수 있다.
- [x] TfidfVectorizer를 적용했다.
- [x] 단어별 IDF 값을 확인했다.
- [x] 특정 도서의 주요 TF-IDF 단어를 확인했다.
- [x] 전체 평균 TF-IDF 상위 단어를 확인했다.
- [x] `chapter03_tfidf_top_terms.csv`를 저장했다.
- [x] Count와 TF-IDF 결과를 비교했다.
- [x] 결과를 원본 제목과 비교했다.
- [x] TF-IDF가 높다는 이유만으로 판매 원인이라고 해석하지 않았다.
- [x] Chapter 04에서는 train 데이터에만 Vectorizer를 fit해야 한다는 점을 이해했다.
- [x] 실제 실행 결과를 바탕으로 Markdown을 작성했다.

---
# 실습 56. 이번 Chapter에서 꼭 기억할 개념

### 1. 텍스트는 머신러닝을 위해 숫자로 변환해야 한다
> 문자열 -> feature -> 숫자 벡터

### 2. Bag of Words는 단어의 등장에 초점을 둔다
단어 순서와 깊은 문맥보다는 **어떤 단어가 몇 번 등장했는지**를 이용한다.

### 3. CountVectorizer의 값은 등장 횟수다
```
행 = 문서 / 열 = 단어 / 값 = 해당 문서에서 그 단어가 나온 횟수
```

### 4. TF-IDF는 전체 문서에서의 흔함도 함께 고려한다
> 문서 안의 빈도 + 전체 문서에서의 희소성 -> TF-IDF 가중치

### 5. 높은 TF-IDF는 현실 세계의 중요도를 의미하지 않는다
현재 문서 집합에서 텍스트 특징으로 **상대적으로 두드러진다**는 의미다.

### 6. fit과 transform을 구분해야 한다
> `fit` = 기준 학습 / `transform` = 학습된 기준으로 변환

### 7. 모델 평가에서는 테스트 데이터 정보가 학습에 들어가면 안 된다
이번 Chapter의 전체 데이터 `fit_transform()`은 **개념 학습용**이다.
Chapter 04에서는 train/test를 먼저 나누고 **train에만 `fit()`** 한다.

---
# 실습 57. Chapter 03 결과물 정리

```
chapter03.ipynb
chapter03_count_top_terms.csv
chapter03_tfidf_top_terms.csv
chapter03_count_matrix.npz   (선택)
chapter03_tfidf_matrix.npz   (선택)
```

## 프롬프트 예시

```
이번 Chapter에서 만들어야 할 결과 파일 4개가 모두 생성되었는지
한 번에 확인하고 싶습니다.
파일 이름과 존재 여부를 나란히 출력해 주세요.
함수로 만들지 말고 바로 실행되는 코드로 작성해 주세요.
try/except 예외 처리는 넣지 마세요.
5줄 이내로 짧게 작성해 주세요.
설명은 코드 옆 짧은 주석으로만 달아 주세요.
```

In [60]:
# 결과물이 모두 만들어졌는지 확인
for name in [
    "chapter03_count_top_terms.csv",
    "chapter03_tfidf_top_terms.csv",
    "chapter03_count_matrix.npz",
    "chapter03_tfidf_matrix.npz",
]:
    print(f"{name:35} {Path(name).exists()}")

chapter03_count_top_terms.csv       True
chapter03_tfidf_top_terms.csv       True
chapter03_count_matrix.npz          True
chapter03_tfidf_matrix.npz          True


---
## Chapter 03 결과 해석

교보문고 베스트셀러 도서 제목 986개를 `CountVectorizer`와 `TfidfVectorizer`로 숫자 벡터로 변환했다.
두 Vectorizer 모두 **986행 × 2,179열**의 행렬을 만들었고, 단어 집합도 동일했다.

CountVectorizer 결과에서 각 행은 하나의 도서 제목, 각 열은 하나의 단어이며
값은 해당 제목에서 단어가 등장한 횟수를 의미한다.
TF-IDF에서는 한 제목에서의 단어 등장뿐 아니라
전체 제목에서 그 단어가 얼마나 흔하게 등장하는지도 함께 반영된다.

실제 상위 단어를 비교한 결과 Count 기준과 평균 TF-IDF 기준의 순위에는 일부 차이가 있었다.
따라서 **단순히 자주 등장하는 단어**와 **각 문서의 특징을 상대적으로 잘 나타내는 단어**는
같은 개념이 아님을 확인했다.

전체 셀 2,148,494개 중 0이 아닌 값은 3,744개로 **0의 비율이 99.8%** 였다.
도서 제목이 짧기 때문에 대부분의 단어가 각 제목에 등장하지 않는 것이며,
이것이 희소 행렬을 사용하는 이유다.

이번 결과는 현재 도서 제목 데이터와 현재 Vectorizer 설정을 기준으로 한 텍스트 특징이며,
단어의 중요도를 **판매 원인이나 독자 선호로 직접 해석하지 않았다.**

### Chapter 02 결과와 다른 점

Chapter 02(Kiwi 형태소 분석)에서는 `해커스`, `기출`, `기념`이 상위였지만,
이번 기본 Vectorizer에서는 `2026`, `2027` 같은 **연도 숫자**가 1·2위로 올라왔다.
기본 토큰 패턴이 숫자를 그대로 단어로 취급하기 때문이다.

**어느 쪽이 틀린 것이 아니라 전처리 조건이 다른 것이다.**

---

## 다음 Chapter 연결

```
Chapter 02  도서 제목에서 단어를 추출하고 빈도를 센다
    ↓
Chapter 03  CountVectorizer와 TF-IDF로 텍스트를 숫자 벡터로 바꾼다
    ↓
Chapter 04  숫자 벡터를 이용해 도서 분야를 분류한다
```

> **텍스트를 숫자로 바꾸는 Vectorizer도 학습 과정의 일부이므로,
> 모델 평가에서는 훈련 데이터에만 fit해야 한다.**